# Simulación Monte Carlo del nanodisco — lista para correr

Sube este archivo a Kaggle, Colab o córrelo en tu máquina. **No necesita el
repositorio**: lleva dentro los tres módulos que usa, con su `sha256`
verificado, así que ejecuta código idéntico byte a byte al del repositorio.

## Qué hace

Enfría un nanodisco magnético desde una temperatura caliente hasta la
temperatura objetivo de cada estado, con Metropolis de un solo sitio sobre un
tablero de ajedrez, y reporta las métricas físicas en cada escalón.

## Cómo usarlo

**Edita sólo la primera celda.** Los parámetros son un arreglo de arreglos: un
arreglo por estado, y el cuadernillo cuenta cuántos hay. Después, *Ejecutar
todo*.

## Qué motor usa

Hay dos, y la celda de configuración elige:

| motor | qué es |
|---|---|
| **estándar** | el de JAX: consigue los seis vecinos creando seis copias desplazadas de la red con `jnp.roll` |
| **optimizado** | un *stencil* escrito a mano en Triton: lee su parcela más un borde, hace todo en registros y escribe una vez — las seis copias nunca existen |

El optimizado corre entre un **57 % y un 79 % más rápido** según el ancho del
lote, y **no se usa sin verificarlo**: antes de cronometrar nada reproduce el
*stencil* del estándar con la misma aleatoriedad (cero espines distintos de
182 520) y compara las dos cadenas a equilibrio.

> Con GPU. Sin GPU corre igual, en CPU, pero mucho más despacio: la escalera
> completa de un estado son unas 31 horas contra 2.9 minutos en una H100.


## 1 · Parámetros

Lo único que hay que editar.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  LOS PARÁMETROS — lo único que hay que editar
# ══════════════════════════════════════════════════════════════════════════
#
# Un arreglo por estado. El cuadernillo cuenta los arreglos y de ahí sale
# cuántos estados hay: no hay que declararlo en ningún otro sitio.
#
# Cada arreglo lleva los OCHO parámetros del conjunto de datos, EN ESTE ORDEN:
#
#   índice  nombre   qué es                                        unidades
#   ──────  ──────   ───────────────────────────────────────────   ────────
#     0     Jex2     intercambio de la 2.ª capa de vecinos        (ver abajo)
#     1     Jex3     intercambio de la 3.ª capa                   (ver abajo)
#     2     Jex4     intercambio de la 4.ª capa                   (ver abajo)
#     3     Kan1     anisotropía cúbica, constante de 1.er orden   meV
#     4     KanS     anisotropía de superficie                     meV
#     5     Hex      campo externo (magnitud)                      meV
#     6     KDM      Dzyaloshinskii–Moriya interfacial             meV
#
# LA TEMPERATURA NO ES UN PARÁMETRO. El enfriamiento es siempre el mismo,
# de 20 K a 0.1 K en pasos de 0.1 K, para todos los estados — son 200
# escalones, y np.linspace(20, 0.1, 200) los da exactos porque 19.9/199 = 0.1.
# Fijarla hace que los estados sean comparables entre sí, y convierte la
# temperatura de TRANSICIÓN en algo que se mide al final en vez de algo que se
# declara al principio.
#
# El intercambio de PRIMEROS vecinos está normalizado a Jex = 1, que es la
# escala de energía del modelo; los demás se dan relativos a él.
#
# LAS CAPAS DE VECINOS Y LAS SUBREDES
#
# Dejar un Jex2, Jex3 o Jex4 distinto de cero enciende esa capa de vecinos, y
# el cuadernillo lo detecta solo: cuenta cuáles están activas y elige el
# coloreo que corresponda. No hay nada más que configurar.
#
# Por qué hace falta elegir. Actualizar varios sitios a la vez es un
# reordenamiento EXACTO del barrido secuencial sólo si ningún par de sitios
# del mismo color interactúa. Con primeros vecinos basta un tablero de dos
# colores, porque todo primer vecino está a un desplazamiento de suma impar y
# por tanto cae en el otro color. Las capas 2 y 4 tienen desplazamientos de
# suma par y rompen eso, así que piden más subredes:
#
#   capas activas        coloreo                        subredes
#   ──────────────────   ────────────────────────────   ────────
#   1, o 1+3             (x+y+z) mod 2                      2
#   cualquiera con la 2  (x mod 2, y mod 2, z mod 2)         8
#   cualquiera con la 4  (x mod 3, y mod 3, z mod 3)        27
#
# La capa 4 llega a distancia 2 sobre un eje, y ningún coloreo de paso 2 puede
# separar eso: de ahí el paso 3. El cuadernillo VERIFICA el coloreo elegido
# recorriendo los desplazamientos de cada capa activa, en vez de darlo por
# bueno.
#
# Lo que cuesta. Un barrido pasa a ser un pase por subred, y cada pase
# actualiza 1/n de los sitios. El trabajo total por barrido no cambia, pero el
# paralelismo disponible en cada pase sí se divide por n — que en una GPU es lo
# que manda. Y el kernel optimizado de Triton está escrito para la primera
# capa: con capas superiores el cuadernillo usa el motor de JAX y lo avisa.

PARAMETROS = [
    # [ Jex2, Jex3, Jex4,  Kan1  , KanS ,  Hex   ,  KDM   ]
    [  0.0,  0.0,  0.0,  0.2867,  0.0 ,  0.7691,  0.6535],  # uniform
    [  0.0,  0.0,  0.0,  0.1770,  0.139,  0.4830,  0.8800],  # helical
    [  0.0,  0.0,  0.0,  0.3500,  0.0 ,  0.2000,  0.9500],  # points
    [  0.0,  0.0,  0.0,  0.1000,  0.200,  0.9000,  0.3000],  # lines
]

# Nombres para las tablas y las figuras. Si se deja vacío se numeran solos.
NOMBRES = ["uniform", "helical", "points", "lines"]

# ══════════════════════════════════════════════════════════════════════════
#  LA GEOMETRÍA
# ══════════════════════════════════════════════════════════════════════════
RD     = 18.30   # radio del disco, en pasos de red. Define N = 2(⌊Rd⌋+1)+1
CAPAS  = 5       # capas en z. ⚠ ver la nota del ancho de vector más abajo
ANGULO = 90.0    # dirección del campo externo, en grados sobre el plano x–z

# ══════════════════════════════════════════════════════════════════════════
#  EL CRONOGRAMA
# ══════════════════════════════════════════════════════════════════════════
REPLICAS      = 50     # réplicas independientes POR ESTADO
T_INICIAL     = 20.0   # inicio del enfriamiento, en K
T_FINAL       = 0.1    # fin del enfriamiento, en K
N_TEMPERATURAS = 200   # escalones: 200 da pasos de exactamente 0.1 K
BARRIDOS      = 20000  # barridos por temperatura (⅔ termalizan, ⅓ miden)
SEMILLA       = 42

# ══════════════════════════════════════════════════════════════════════════
#  EL MOTOR
# ══════════════════════════════════════════════════════════════════════════
# "auto"       -> usa el kernel optimizado de Triton si se puede, si no el de JAX
# "optimizado" -> exige Triton; falla si no está disponible
# "estandar"   -> el motor de JAX de siempre, sin Triton
MOTOR = "auto"

# ⚠  SOBRE EL NÚMERO DE RÉPLICAS Y DE CAPAS
#    Con el motor "estandar" el costo depende de la divisibilidad por dos:
#    hay castigo (hasta +49 % según la tarjeta) cuando
#        v₂(REPLICAS) + v₂(CAPAS) ≥ 2
#    siendo v₂(n) cuántas veces n se divide por 2. Con CAPAS = 5 (impar) basta
#    con que REPLICAS no sea múltiplo de cuatro. El motor "optimizado" NO paga
#    ese castigo (+0.3 % medido), así que ahí el número es libre.
#    El cuadernillo avisa cuál es tu caso antes de arrancar.


## 2 · Entorno

Detecta la GPU, instala lo que falte y deja coherente la cadena de herramientas.

In [ ]:
# Entorno: detecta la GPU, instala lo que falte y deja la cadena de
# herramientas coherente. Pensado para Kaggle, Colab o una máquina propia.
import subprocess, sys, os, glob, re

def _sh(*a):
    return subprocess.run(a, capture_output=True, text=True).stdout.strip()

print("GPU:", _sh("nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                  "--format=csv,noheader") or "no se detectó ninguna")

def _tiene(mod):
    try:
        __import__(mod); return True
    except Exception:
        return False

faltan = [p for p, m in (("jax[cuda12]", "jax"), ("jax-triton", "jax_triton"))
          if not _tiene(m)]
if faltan:
    print("instalando:", " ".join(faltan))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan],
                   check=False)

# Triton trae su propio ptxas, a veces una versión de CUDA por delante del que
# usa XLA para ensamblar. Si difieren, nada compila. Se fija Triton al de XLA
# ANTES de importar jax, porque el backend congela su cadena al inicializarse.
def _ver(p):
    m = re.search(r"release (\d+)\.(\d+)", _sh(p, "--version"))
    return tuple(map(int, m.groups())) if m else (0, 0)

_pats = ["/usr/local/lib/python3*/dist-packages/nvidia/cuda_nvcc/bin/ptxas",
         "/usr/local/lib/python3*/site-packages/nvidia/cuda_nvcc/bin/ptxas",
         "/opt/conda/lib/python3*/site-packages/nvidia/cuda_nvcc/bin/ptxas",
         "/usr/local/cuda*/bin/ptxas"]
_xla = sorted({p for q in _pats for p in glob.glob(q)})
_xla = [(v, p) for v, p in ((_ver(p), p) for p in _xla) if v > (0, 0)]
if _xla:
    v, p = min(_xla)
    os.environ["TRITON_PTXAS_PATH"] = p
    print(f"TRITON_PTXAS_PATH -> CUDA {v[0]}.{v[1]}")

import jax
print("jax", jax.__version__, "|", jax.devices())
try:
    import triton, jax_triton as jt
    print("triton", triton.__version__, "| jax_triton",
          getattr(jt, "__version__", "?"))
except Exception as e:
    print("Triton no disponible:", type(e).__name__, e)


## 3 · Los módulos

Se escriben a disco desde los literales de abajo, con su `sha256` verificado.

In [ ]:
# sim_core.py — idéntico byte a byte al del repositorio. sha256 649973371a8206cf4239a4f1dd3f1d0b0c3a7495755d282dd6e1e2cc86e10b2c
import hashlib
_src = '"""Monte Carlo engine for the nanodisk, faithful to the project\'s reference notebook.\n\nTaken from `00_benchmark_mc_observables.ipynb` -- the same checkerboard, the same\nuniform-sphere proposal, and in particular the same DMI: **in-plane bonds only\n(x and y), with a minus sign**. That is interfacial DMI, the physics of a thin\nfilm whose DM vector comes from an interface, and it is not interchangeable with\nthe all-axes form: on the same configuration the total energy differs by 50%.\n\nTwo terms the reference notebook does not carry are added here because the\ntutor\'s dataset varies them:\n\n- `KanS`, the surface anisotropy, applied where a site has fewer than six\n  in-disk nearest neighbours (the coordination test used in the group\'s\n  multi-GPU notebooks).\n- `Kan2`, read as the SECOND-ORDER constant of the cubic anisotropy expansion,\n\n      E_an = (Kan1 / So^4)(sx^2 sy^2 + sx^2 sz^2 + sy^2 sz^2)\n           + (Kan2 / So^6)(sx^2 sy^2 sz^2)\n\n  which is the textbook K1/K2 series and matches the naming. It is a single-site\n  term, so unlike a second-neighbour coupling it leaves the checkerboard exact.\n"""\n\nfrom __future__ import annotations\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\nfrom jax import random\n\n\ndef build_geometry(Rd: float = 18.30, L: int = 5):\n    """Disk mask, checkerboard sublattices and the surface mask."""\n    L2 = int(Rd) + 1\n    N = 2 * L2 + 1\n    x = np.arange(-L2, L2 + 1)\n    xg, yg = np.meshgrid(x, x, indexing="xy")\n    disk2d = (xg ** 2 + yg ** 2) <= Rd ** 2\n    disco = np.broadcast_to(disk2d, (L, N, N)).copy()\n\n    zc, yc, xc = np.indices((L, N, N))\n    checker = (xc + yc + zc) % 2\n    even = (checker == 0) & disco\n    odd = (checker == 1) & disco\n\n    # Surface = fewer than six in-disk nearest neighbours. x and y wrap, z does\n    # not, so the top and bottom layers are surface everywhere.\n    surface = np.zeros_like(disco, dtype=bool)\n    for z in range(L):\n        for y in range(N):\n            for xx in range(N):\n                if not disco[z, y, xx]:\n                    continue\n                n = 0\n                for dz, dy, dx in ((0, 0, 1), (0, 0, -1), (0, 1, 0),\n                                   (0, -1, 0), (1, 0, 0), (-1, 0, 0)):\n                    nz, ny, nx = z + dz, (y + dy) % N, (xx + dx) % N\n                    if 0 <= nz < L and disco[nz, ny, nx]:\n                        n += 1\n                if n < 6:\n                    surface[z, y, xx] = True\n    return dict(L=L, N=N, Rd=Rd, disco=disco, even=even, odd=odd,\n                surface=surface, disk2d=disk2d, n_active=int(disco.sum()),\n                n_surface=int(surface.sum()))\n\n\ndef build_engine(geo, So=1.0, with_mvec=False):\n    """Returns (substep, energy, magnetization, topological_charge, anneal)."""\n    L, N = geo["L"], geo["N"]\n    even_j = jnp.asarray(geo["even"][None, ..., None])\n    odd_j = jnp.asarray(geo["odd"][None, ..., None])\n    disco_j = jnp.asarray(geo["disco"][None, ..., None])\n    surf_j = jnp.asarray(geo["surface"][None, ...])\n    disk2d_j = jnp.asarray(geo["disk2d"])\n    n_act = geo["n_active"]\n\n    def _an(s, Kan1, Kan2, K1_eff):\n        """Cubic anisotropy: the K1 invariant plus the K2 (sixth-order) one."""\n        sx, sy, sz = s[..., 0], s[..., 1], s[..., 2]\n        e = (K1_eff / So ** 4) * ((sx * sy) ** 2 + (sx * sz) ** 2 + (sy * sz) ** 2)\n        return e + (Kan2 / So ** 6) * (sx * sy * sz) ** 2\n\n    def substep(spins, key, beta, mask, p, K1_eff):\n        S_xp = jnp.roll(spins, -1, 3); S_xm = jnp.roll(spins, 1, 3)\n        S_yp = jnp.roll(spins, -1, 2); S_ym = jnp.roll(spins, 1, 2)\n        S_zp = jnp.roll(spins, -1, 1).at[:, -1].set(0.0)\n        S_zm = jnp.roll(spins, 1, 1).at[:, 0].set(0.0)\n\n        kt, ka = random.split(key)\n        tr = random.normal(kt, spins.shape)\n        tr = tr / jnp.linalg.norm(tr, axis=-1, keepdims=True) * So\n        tr = jnp.where(mask, tr, spins)\n        dS = tr - spins\n\n        sv = S_xp + S_xm + S_yp + S_ym + S_zp + S_zm\n        dE = -p["Jex"] * jnp.sum(dS * sv, axis=-1)\n        dE = dE + _an(tr, p["Kan1"], p["Kan2"], K1_eff) - _an(spins, p["Kan1"], p["Kan2"], K1_eff)\n        # Interfacial DMI: in-plane bonds only, minus sign. See module docstring.\n        dE = dE - p["KDM"] * (dS[..., 1] * (S_xp[..., 2] - S_xm[..., 2])\n                              - dS[..., 2] * (S_xp[..., 1] - S_xm[..., 1])\n                              + dS[..., 2] * (S_yp[..., 0] - S_ym[..., 0])\n                              - dS[..., 0] * (S_yp[..., 2] - S_ym[..., 2]))\n        dE = dE - p["Hex"] * (dS[..., 0] * jnp.cos(p["g"]) + dS[..., 2] * jnp.sin(p["g"]))\n\n        r = random.uniform(ka, dE.shape)\n        acc = ((dE < 0) | (r < jnp.exp(-dE * beta)))[..., None] & mask\n        return jnp.where(acc, tr, spins)\n\n    def energy(spins, p, K1_eff):\n        """Total energy; each bond counted once via the +x, +y, +z shifts only."""\n        S_xp = jnp.roll(spins, -1, 3)\n        S_yp = jnp.roll(spins, -1, 2)\n        S_zp = jnp.roll(spins, -1, 1).at[:, -1].set(0.0)\n        E = -p["Jex"] * jnp.sum(spins * (S_xp + S_yp + S_zp), axis=-1)\n        sx, sy, sz = spins[..., 0], spins[..., 1], spins[..., 2]\n        E = E + _an(spins, p["Kan1"], p["Kan2"], K1_eff)\n        E = E - p["KDM"] * (sy * S_xp[..., 2] - sz * S_xp[..., 1]\n                            + sz * S_yp[..., 0] - sx * S_yp[..., 2])\n        E = E - p["Hex"] * (sx * jnp.cos(p["g"]) + sz * jnp.sin(p["g"]))\n        return jnp.sum(E * disco_j[..., 0], axis=(-3, -2, -1))\n\n    def mag_vec(spins):\n        """The magnetisation VECTOR, which `magnetization` collapses."""\n        return jnp.sum(spins * disco_j, axis=(1, 2, 3)) / n_act\n\n    def magnetization(spins):\n        M = jnp.sum(spins * disco_j, axis=(1, 2, 3)) / n_act\n        return jnp.linalg.norm(M, axis=-1), M[:, 2]\n\n    def _omega(s1, s2, s3):\n        num = jnp.sum(s1 * jnp.cross(s2, s3), axis=-1)\n        den = (1.0 + jnp.sum(s1 * s2, -1) + jnp.sum(s2 * s3, -1) + jnp.sum(s3 * s1, -1))\n        return 2.0 * jnp.arctan2(num, den)\n\n    def topo_charge(spins):\n        """Berg-Luscher charge of every layer: quantised, unlike a finite difference."""\n        s00 = spins[..., :-1, :-1, :]; s10 = spins[..., :-1, 1:, :]\n        s01 = spins[..., 1:, :-1, :];  s11 = spins[..., 1:, 1:, :]\n        valid = (disk2d_j[:-1, :-1] & disk2d_j[:-1, 1:]\n                 & disk2d_j[1:, :-1] & disk2d_j[1:, 1:])\n        w = _omega(s00, s10, s11) + _omega(s00, s11, s01)\n        return jnp.sum(jnp.where(valid, w, 0.0), axis=(-2, -1)) / (4.0 * jnp.pi)\n\n    def anneal(spins, key, betas, p, n_therm, n_meas):\n        K1_eff = jnp.where(surf_j, p["KanS"], p["Kan1"])\n\n        def per_T(carry, beta):\n            s, k = carry\n\n            def therm(_, v):\n                s, k = v\n                k, k1, k2 = random.split(k, 3)\n                s = substep(s, k1, beta, even_j, p, K1_eff)\n                s = substep(s, k2, beta, odd_j, p, K1_eff)\n                return s, k\n            s, k = jax.lax.fori_loop(0, n_therm, therm, (s, k))\n\n            b = s.shape[0]\n            z0 = jnp.zeros(b)\n\n            # C_v and chi are variances, and accumulating raw E and E^2\n            # destroys them in float32. At T = 0.2 this run has <E> ~ -1.7e4, so\n            # <E^2> ~ 2.9e8, where one float32 ulp is ~34 -- larger than the\n            # variance itself. <E^2> - <E>^2 then comes out NEGATIVE and C_v\n            # reads -437: not a small error but pure cancellation.\n            #\n            # Measuring the fluctuation about a reference taken once, before the\n            # loop, keeps every accumulated quantity at the scale of the\n            # fluctuation rather than of the total. A constant shift leaves the\n            # variance unchanged, so this is exact. Both the centred moments and\n            # the reference are returned, and `reduce_obs` recombines them.\n            E_ref = energy(s, p, K1_eff)\n            M_ref, _ = magnetization(s)\n\n            zv = jnp.zeros((b, 3))\n\n            def meas(_, v):\n                s, k, sE, sE2, sM, sM2, sMz, sQ, sMv = v\n                k, k1, k2 = random.split(k, 3)\n                s = substep(s, k1, beta, even_j, p, K1_eff)\n                s = substep(s, k2, beta, odd_j, p, K1_eff)\n                dE = energy(s, p, K1_eff) - E_ref\n                M, Mz = magnetization(s)\n                dM = M - M_ref\n                Q = topo_charge(s)[:, L // 2]\n                return (s, k, sE + dE, sE2 + dE * dE,\n                        sM + dM, sM2 + dM * dM, sMz + Mz, sQ + Q,\n                        sMv + mag_vec(s))\n\n            s, k, sE, sE2, sM, sM2, sMz, sQ, sMv = jax.lax.fori_loop(\n                0, n_meas, meas, (s, k, z0, z0, z0, z0, z0, z0, zv))\n            obs = jnp.stack([sE / n_meas, sE2 / n_meas,\n                             sM / n_meas, sM2 / n_meas,\n                             sMz / n_meas, sQ / n_meas, E_ref, M_ref])\n            # One middle-layer snapshot per temperature, replica 0 only. The\n            # whole batch would be 30 x 100 x 5 x 39 x 39 x 3 floats (~2.7 GB);\n            # one replica\'s middle layer is 182 kB and is what the state-\n            # transition figure needs.\n            snap = s[0, L // 2, :, :, 2]\n            if with_mvec:\n                return (s, k), (obs, snap, sMv / n_meas)\n            return (s, k), (obs, snap)\n\n        if with_mvec:\n            (s, _), (all_obs, snaps, mvec) = jax.lax.scan(\n                per_T, (spins, key), betas)\n            return all_obs, s, snaps, mvec\n        (s, _), (all_obs, snaps) = jax.lax.scan(per_T, (spins, key), betas)\n        return all_obs, s, snaps\n\n    return dict(substep=substep, energy=energy, magnetization=magnetization,\n                mag_vec=mag_vec, topo_charge=topo_charge, anneal=anneal)\n\n\ndef init_spins(key, geo, batch, So=1.0):\n    L, N = geo["L"], geo["N"]\n    x = random.normal(key, (batch, L, N, N, 3))\n    x = x / jnp.linalg.norm(x, axis=-1, keepdims=True) * So\n    return x * jnp.asarray(geo["disco"][None, ..., None])\n\n\ndef reduce_obs(obs, temps, n_spins, kB=0.086173404):\n    """Accumulators -> per-temperature scalars.\n\n    `lax.scan` stacks each step\'s output along a NEW LEADING axis, so `anneal`\n    returns `(n_T, 6, batch)` and the observable index is the SECOND axis. The\n    obvious `obs[i]` indexes temperatures instead, silently mixing the two --\n    and a test with six temperatures and six observables cannot tell the\n    difference, since both shapes read `(6, 6, batch)`.\n    """\n    obs = np.asarray(obs)\n    assert obs.ndim == 3 and obs.shape[1] == 8, (\n        f"expected (n_T, 8, batch) from lax.scan, got {obs.shape}")\n    dE, dE2, dM, dM2, Mz, Q, E_ref, M_ref = [obs[:, i] for i in range(8)]\n    T = np.asarray(temps)[:, None]\n    # Variances of the CENTRED quantities: both terms sit at the scale of the\n    # fluctuation, so there is nothing to cancel.\n    var_E = dE2 - dE ** 2\n    var_M = dM2 - dM ** 2\n    E = dE + E_ref\n    M = dM + M_ref\n    Cv = var_E / (kB * T ** 2 * n_spins)\n    chi = var_M / (kB * T)\n    return dict(T=np.asarray(temps),\n                E=E.mean(1) / n_spins, E_sd=E.std(1) / n_spins,\n                M=M.mean(1), M_sd=M.std(1),\n                Mz=Mz.mean(1), Cv=Cv.mean(1), chi=chi.mean(1),\n                Q=Q.mean(1), Q_sd=Q.std(1))\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "649973371a8206cf4239a4f1dd3f1d0b0c3a7495755d282dd6e1e2cc86e10b2c", "sim_core.py alterado"
open("sim_core.py", "w").write(_src)
print("sim_core.py  {} bytes  sha256 ok".format(len(_src)))


In [ ]:
# triton_halo.py — idéntico byte a byte al del repositorio. sha256 f397f10c5870a42bfe273a101f5bbada5a79ad6259d140cf636ee51584ca5a57
import hashlib
_src = '"""A raw-Triton substep that keeps the neighbourhood in registers.\n\nWhy this exists\n---------------\nThe JAX substep builds six shifted copies of the lattice with `jnp.roll` and\nthen does arithmetic on them. HLO attributes 31.5% of the anneal to those six\nrolls alone. Three earlier attempts to remove them went through Pallas and all\nlost, by +15% to +31%, for one reason recorded in\n`the-cost-is-the-gather-not-the-lanes`: each lane computed its own neighbour\naddresses, so the compiler could no longer prove the accesses were contiguous\nand emitted scattered loads. 1521 sites x 7 neighbours x 3 components is about\n32k address computations per site-step, against seven coalesced array reads in\nthe JAX path.\n\nThis kernel is the version that was never writable in Pallas. The difference\nis not fewer loads -- it still issues 21 of them -- but their SHAPE:\n\n    Pallas gather   : one address per lane, computed per lane  -> scattered\n    this kernel     : one contiguous run per row, shifted      -> coalesced\n\nA shifted contiguous load is the same access pattern a roll produces, except\nthe shifted plane is never materialised. It lands in registers, the whole\nupdate happens there, and only the result is stored. So the six rolls\'\nround-trip through memory disappears while the coalescing survives. In Triton\nterms that is "the halo in SRAM": the overlapping shifted loads of adjacent\nblocks hit L1/L2, and the stencil never writes an intermediate.\n\nWhy the halo needs no coordination: the update is a checkerboard, so every\nneighbour of an updated site has the OPPOSITE parity and no block writes it\nduring this substep. A block can therefore read its halo straight from global\nmemory with no barrier -- which is the whole reason the checkerboard exists.\n\nLayout\n------\nProduction spins are `(B, L, N, N, 3)`, component-minor, so a run along x has\nstride 3. This kernel takes `(B, 3, L, N, N)`, component-planar, where a run\nalong x is stride 1 and a row load is one contiguous 39-float segment. The\ntranspose is paid once outside the step loop, so over thousands of steps it\ncosts nothing.\n\nTwo variants, for two different jobs\n------------------------------------\n`HAS_RNG=False` takes the trial spins and the Metropolis uniforms as inputs,\nso it can be compared against `sim_core.substep` on identical randomness --\nthat validates the stencil. `HAS_RNG=True` draws them in-kernel with Philox,\nwhich is the real candidate: it also absorbs the 17.1% of the anneal that\nHLO attributes to Threefry.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\nimport triton\nimport triton.language as tl\n\n\n# ---------------------------------------------------------------- the kernel\n\n@triton.jit\ndef _substep_kernel(\n    sp_ptr,          # (B, 3, LZ, N, N)  f32   spins in\n    k1_ptr,          # (B, LZ, N, N)     f32   K1_eff, surface-dependent\n    par_ptr,         # (B, 6)            f32   Jex, Kan2, Hex, KDM, cos g, sin g\n    disk_ptr,        # (N, N)            f32   1.0 inside the disk\n    beta_ptr,        # (1,)              f32\n    step_ptr,        # (1,)              i32   mixes into the Philox seed\n    tr_ptr,          # (B, 3, LZ, N, N)  f32   trial spins   (HAS_RNG=False)\n    ru_ptr,          # (B, LZ, N, N)     f32   uniforms      (HAS_RNG=False)\n    out_ptr,         # (B, 3, LZ, N, N)  f32   spins out\n    de_ptr,          # (B, LZ, N, N)     f32   dE out        (WRITE_DE=True)\n    N: tl.constexpr,\n    LZ: tl.constexpr,\n    PARITY: tl.constexpr,     # 0 updates the even sublattice, 1 the odd\n    BY: tl.constexpr,         # rows of the block\n    BX: tl.constexpr,         # lanes along x, a power of two >= N\n    HAS_RNG: tl.constexpr,\n    WRITE_DE: tl.constexpr,\n    SO: tl.constexpr,\n):\n    b = tl.program_id(0)\n    z = tl.program_id(1)\n    ty = tl.program_id(2)\n\n    NN = N * N\n    sb = 3 * LZ * NN          # stride between replicas\n    sc = LZ * NN              # stride between components\n\n    # ---- the block, and its neighbour index vectors.\n    # These are VECTORS over the block, not per-lane addresses: `xc + 1` is a\n    # contiguous run shifted by one element, which is what keeps the loads\n    # coalesced. That is the single difference from the gather attempts.\n    ys = ty * BY + tl.arange(0, BY)\n    xs = tl.arange(0, BX)\n    live = (ys < N)[:, None] & (xs < N)[None, :]\n\n    yc = tl.where(ys < N, ys, 0)\n    xc = tl.where(xs < N, xs, 0)\n\n    # x and y wrap; z does not (open boundary, handled by the gates below).\n    ypi = tl.where(yc + 1 >= N, 0, yc + 1)\n    ymi = tl.where(yc - 1 < 0, N - 1, yc - 1)\n    xpi = tl.where(xc + 1 >= N, 0, xc + 1)\n    xmi = tl.where(xc - 1 < 0, N - 1, xc - 1)\n\n    o_c = yc[:, None] * N + xc[None, :]\n    o_yp = ypi[:, None] * N + xc[None, :]\n    o_ym = ymi[:, None] * N + xc[None, :]\n    o_xp = yc[:, None] * N + xpi[None, :]\n    o_xm = yc[:, None] * N + xmi[None, :]\n\n    zp = tl.minimum(z + 1, LZ - 1)\n    zm = tl.maximum(z - 1, 0)\n    # `jnp.roll(...).at[:, -1].set(0.0)` in the reference: the plane past the\n    # last layer contributes zero rather than wrapping.\n    gp = tl.where(z < LZ - 1, 1.0, 0.0)\n    gm = tl.where(z > 0, 1.0, 0.0)\n\n    # ---- the site mask: inside the disk, and on the sublattice being updated.\n    disk = tl.load(disk_ptr + o_c, mask=live, other=0.0)\n    par = (xc[None, :] + yc[:, None] + z) % 2\n    upd = live & (disk > 0.5) & (par == PARITY)\n\n    # ---- per-chain parameters.\n    Jex = tl.load(par_ptr + b * 6 + 0)\n    Kan2 = tl.load(par_ptr + b * 6 + 1)\n    Hex = tl.load(par_ptr + b * 6 + 2)\n    KDM = tl.load(par_ptr + b * 6 + 3)\n    cosg = tl.load(par_ptr + b * 6 + 4)\n    sing = tl.load(par_ptr + b * 6 + 5)\n    beta = tl.load(beta_ptr)\n\n    sca = sp_ptr + b * sb\n    k1 = tl.load(k1_ptr + b * LZ * NN + z * NN + o_c, mask=live, other=0.0)\n\n    # ---- the spin at the site.\n    s0 = tl.load(sca + 0 * sc + z * NN + o_c, mask=live, other=0.0)\n    s1 = tl.load(sca + 1 * sc + z * NN + o_c, mask=live, other=0.0)\n    s2 = tl.load(sca + 2 * sc + z * NN + o_c, mask=live, other=0.0)\n\n    # ---- the trial spin: a uniform point on the sphere.\n    if HAS_RNG:\n        off = ((b * LZ + z) * N + yc[:, None]) * N + xc[None, :]\n        # One key, four distinct COUNTERS -- the textbook use of a\n        # counter-based generator. An earlier version varied the KEY instead\n        # (`sd+0..3` at a shared counter) and the engine then accepted 1.1%\n        # less than production, systematically across five temperatures,\n        # while its proposals were provably isotropic and unit-norm. That\n        # leaves the uniform: adjacent keys at one counter let `ru` correlate\n        # with the proposed direction, hence with dE, which biases the\n        # Metropolis test. Separating the counters removes the shared input.\n        sd = tl.load(step_ptr)\n        o4 = off * 4\n        g0 = tl.randn(sd, o4 + 0)\n        g1 = tl.randn(sd, o4 + 1)\n        g2 = tl.randn(sd, o4 + 2)\n        nrm = tl.sqrt(g0 * g0 + g1 * g1 + g2 * g2) + 1e-30\n        t0 = g0 / nrm * SO\n        t1 = g1 / nrm * SO\n        t2 = g2 / nrm * SO\n        ru = tl.rand(sd, o4 + 3)\n    else:\n        trb = tr_ptr + b * sb\n        t0 = tl.load(trb + 0 * sc + z * NN + o_c, mask=live, other=0.0)\n        t1 = tl.load(trb + 1 * sc + z * NN + o_c, mask=live, other=0.0)\n        t2 = tl.load(trb + 2 * sc + z * NN + o_c, mask=live, other=0.0)\n        ru = tl.load(ru_ptr + b * LZ * NN + z * NN + o_c, mask=live, other=0.0)\n\n    d0 = t0 - s0\n    d1 = t1 - s1\n    d2 = t2 - s2\n\n    # ---- the stencil, one component at a time so the six neighbour blocks of\n    # a component die before the next component\'s are loaded. Holding all 21\n    # at once spills at BY >= 8.\n    p0 = sca + 0 * sc\n    n_xp = tl.load(p0 + z * NN + o_xp, mask=live, other=0.0)\n    n_xm = tl.load(p0 + z * NN + o_xm, mask=live, other=0.0)\n    n_yp = tl.load(p0 + z * NN + o_yp, mask=live, other=0.0)\n    n_ym = tl.load(p0 + z * NN + o_ym, mask=live, other=0.0)\n    sv0 = (n_xp + n_xm + n_yp + n_ym\n           + gp * tl.load(p0 + zp * NN + o_c, mask=live, other=0.0)\n           + gm * tl.load(p0 + zm * NN + o_c, mask=live, other=0.0))\n    dyx = n_yp - n_ym                     # DMI needs this difference only\n\n    p1 = sca + 1 * sc\n    n_xp = tl.load(p1 + z * NN + o_xp, mask=live, other=0.0)\n    n_xm = tl.load(p1 + z * NN + o_xm, mask=live, other=0.0)\n    n_yp = tl.load(p1 + z * NN + o_yp, mask=live, other=0.0)\n    n_ym = tl.load(p1 + z * NN + o_ym, mask=live, other=0.0)\n    sv1 = (n_xp + n_xm + n_yp + n_ym\n           + gp * tl.load(p1 + zp * NN + o_c, mask=live, other=0.0)\n           + gm * tl.load(p1 + zm * NN + o_c, mask=live, other=0.0))\n    dxy = n_xp - n_xm\n\n    p2 = sca + 2 * sc\n    n_xp = tl.load(p2 + z * NN + o_xp, mask=live, other=0.0)\n    n_xm = tl.load(p2 + z * NN + o_xm, mask=live, other=0.0)\n    n_yp = tl.load(p2 + z * NN + o_yp, mask=live, other=0.0)\n    n_ym = tl.load(p2 + z * NN + o_ym, mask=live, other=0.0)\n    sv2 = (n_xp + n_xm + n_yp + n_ym\n           + gp * tl.load(p2 + zp * NN + o_c, mask=live, other=0.0)\n           + gm * tl.load(p2 + zm * NN + o_c, mask=live, other=0.0))\n    dxz = n_xp - n_xm\n    dyz = n_yp - n_ym\n\n    # ---- the energy difference, term for term as in `sim_core.substep`.\n    dE = -Jex * (d0 * sv0 + d1 * sv1 + d2 * sv2)\n\n    i4 = 1.0 / (SO * SO * SO * SO)\n    i6 = i4 / (SO * SO)\n    an_t = (k1 * i4 * ((t0 * t1) * (t0 * t1) + (t0 * t2) * (t0 * t2)\n                       + (t1 * t2) * (t1 * t2))\n            + Kan2 * i6 * (t0 * t1 * t2) * (t0 * t1 * t2))\n    an_s = (k1 * i4 * ((s0 * s1) * (s0 * s1) + (s0 * s2) * (s0 * s2)\n                       + (s1 * s2) * (s1 * s2))\n            + Kan2 * i6 * (s0 * s1 * s2) * (s0 * s1 * s2))\n    dE += an_t - an_s\n\n    # Interfacial DMI: in-plane bonds only, with the minus sign.\n    dE -= KDM * (d1 * dxz - d2 * dxy + d2 * dyx - d0 * dyz)\n    dE -= Hex * (d0 * cosg + d2 * sing)\n\n    acc = ((dE < 0.0) | (ru < tl.exp(-dE * beta))) & upd\n\n    ob = out_ptr + b * sb + z * NN + o_c\n    tl.store(ob + 0 * sc, tl.where(acc, t0, s0), mask=live)\n    tl.store(ob + 1 * sc, tl.where(acc, t1, s1), mask=live)\n    tl.store(ob + 2 * sc, tl.where(acc, t2, s2), mask=live)\n    if WRITE_DE:\n        tl.store(de_ptr + b * LZ * NN + z * NN + o_c, dE, mask=live)\n\n\n# ------------------------------------------------------------- JAX plumbing\n\ndef planar(x):\n    """(B, L, N, N, 3) -> (B, 3, L, N, N), paid once outside the step loop.\n\n    No explicit contiguity call is needed: `jax_triton` pins default row-major\n    layouts on its operands, so XLA has to materialise the transpose before\n    the custom call rather than passing a strided view.\n    """\n    import jax.numpy as jnp\n    return jnp.transpose(x, (0, 4, 1, 2, 3))\n\n\ndef unplanar(x):\n    import jax.numpy as jnp\n    return jnp.transpose(x, (0, 2, 3, 4, 1))\n\n\ndef pack_params(p, batch):\n    """The six scalars the kernel reads, as (B, 6).\n\n    `Kan1` is absent on purpose: `sim_core._an` takes it as an argument and\n    never uses it -- the K1 invariant is carried entirely by `K1_eff`, which\n    is already the surface-resolved per-site field.\n    """\n    import jax.numpy as jnp\n    def col(v):\n        a = jnp.asarray(v, dtype=jnp.float32).reshape(-1)\n        return jnp.broadcast_to(a, (batch,)) if a.size == 1 else a\n    g = col(p["g"])\n    return jnp.stack([col(p["Jex"]), col(p["Kan2"]), col(p["Hex"]),\n                      col(p["KDM"]), jnp.cos(g), jnp.sin(g)], axis=1)\n\n\ndef build(geo, BY=4, BX=None, num_warps=4, So=1.0):\n    """Returns `substep(spins_planar, k1, par, beta, step, parity, tr, ru)`.\n\n    `spins_planar` is component-planar. `tr`/`ru` are `None` for the in-kernel\n    RNG variant. `de` comes back only when `want_de` is set, for validation.\n    """\n    import jax\n    import jax.numpy as jnp\n    import jax_triton as jt\n\n    N, LZ = int(geo["N"]), int(geo["L"])\n    BX = int(BX) if BX else 1 << (N - 1).bit_length()\n    BY = int(BY)\n    n_strip = -(-N // BY)\n    disk = jnp.asarray(geo["disk2d"].astype(np.float32))\n\n    def substep(sp, k1, par, beta, step, parity, tr=None, ru=None,\n                want_de=False):\n        B = sp.shape[0]\n        has_rng = tr is None\n        shape = jax.ShapeDtypeStruct(sp.shape, jnp.float32)\n        # The kernel\'s argument list has to stay FIXED: `triton_call` appends\n        # one pointer per entry of `out_shape`, so dropping the dE output left\n        # `de_ptr` unbound and Triton failed to compile the signature. The\n        # output is always declared and shrinks to one element when unwanted.\n        de_shape = jax.ShapeDtypeStruct(\n            (B, LZ, N, N) if want_de else (1,), jnp.float32)\n        outs = (shape, de_shape)\n        # Likewise the trial/uniform inputs: unread under HAS_RNG, so they\n        # shrink to one element instead of allocating a full lattice of zeros\n        # on every iteration of the step loop.\n        dum = jnp.zeros((1,), jnp.float32)\n        res = jt.triton_call(\n            sp, k1, par, disk,\n            jnp.asarray(beta, jnp.float32).reshape(1),\n            jnp.asarray(step, jnp.int32).reshape(1),\n            dum if has_rng else tr,\n            dum if has_rng else ru,\n            kernel=_substep_kernel,\n            out_shape=outs,\n            grid=(B, LZ, n_strip),\n            num_warps=int(num_warps),\n            N=N, LZ=LZ, PARITY=int(parity), BY=BY, BX=BX,\n            HAS_RNG=has_rng, WRITE_DE=bool(want_de), SO=float(So),\n        )\n        return res if want_de else res[0]\n\n    substep.meta = dict(N=N, LZ=LZ, BY=BY, BX=BX, n_strip=n_strip,\n                        num_warps=int(num_warps),\n                        grid_per_replica=LZ * n_strip,\n                        lanes_per_block=BY * BX,\n                        lane_waste_pct=100.0 * (1.0 - (N * N) / float(\n                            BX * BY * n_strip)))\n    return substep\n\n\ndef sweeps(substep, k1, par, beta, n, base_step=0):\n    """`n` sweeps in one XLA program: even then odd, inside a device loop.\n\n    The loop has to live in `lax.fori_loop`. A Python loop would pay the\n    ~122 us host dispatch of a `triton_call` per substep, which is what faked\n    an earlier Triton win and reversed on amortisation.\n    """\n    import jax\n    import jax.numpy as jnp\n\n    def body(i, sp):\n        st = base_step + 2 * i\n        sp = substep(sp, k1, par, beta, st, 0)\n        return substep(sp, k1, par, beta, st + 1, 1)\n\n    def run(sp):\n        return jax.lax.fori_loop(0, int(n), body, sp)\n    return run\n\n\ndef sweeps_extrng(substep, k1, par, beta, n, shape, base_step=0, So=1.0):\n    """`n` sweeps with the trial spins drawn by JAX, not by the kernel.\n\n    This arm exists to split the win. The in-kernel variant changes two things\n    at once -- the stencil and the random source -- so on its own it cannot\n    say which one paid. Here the stencil is the Triton one while the proposals\n    still come from JAX\'s Threefry, exactly as production draws them, so the\n    difference against production isolates the stencil and the difference\n    against the in-kernel variant isolates the RNG.\n\n    `shape` is the component-planar spin shape; drawing the normals directly\n    in that layout and normalising along the component axis is the same\n    distribution as drawing them component-minor and transposing.\n    """\n    import jax\n    import jax.numpy as jnp\n\n    B, _, LZ, N, _ = shape\n\n    def half(sp, k, parity):\n        kt, ka = jax.random.split(k)\n        tr = jax.random.normal(kt, shape)\n        tr = tr / jnp.linalg.norm(tr, axis=1, keepdims=True) * So\n        ru = jax.random.uniform(ka, (B, LZ, N, N))\n        return substep(sp, k1, par, beta, 0, parity, tr, ru)\n\n    def body(i, v):\n        sp, k = v\n        k, k0, k1_ = jax.random.split(k, 3)\n        sp = half(sp, k0, 0)\n        return half(sp, k1_, 1), k\n\n    def run(sp):\n        return jax.lax.fori_loop(0, int(n), body,\n                                 (sp, jax.random.PRNGKey(base_step)))[0]\n    return run\n\n\n# ------------------------------------------- observables, component-planar\n#\n# These exist because of where the measurement loop spends its time. The\n# kernel works on `(B, 3, L, N, N)` while `sim_core`\'s observables expect\n# `(B, L, N, N, 3)`, and transposing once per measured sweep would copy the\n# whole lattice as often as the substep touches it -- enough to eat the win.\n# So the observables are rewritten against the component axis at position 1.\n# Every formula is the one in `sim_core`, term for term; only the axes move.\n\ndef build_obs(geo):\n    import jax.numpy as jnp\n\n    LZ, N = int(geo["L"]), int(geo["N"])\n    disco = jnp.asarray(geo["disco"].astype(np.float32))          # (L, N, N)\n    disk2d = jnp.asarray(geo["disk2d"])\n    n_act = float(geo["n_active"])\n\n    def c4(par, k):\n        """One per-chain scalar, shaped to broadcast against (B, L, N, N)."""\n        return par[:, k].reshape(-1, 1, 1, 1)\n\n    def energy(sp, par, k1):\n        # Each bond counted once, via the +x, +y, +z shifts only.\n        xp = jnp.roll(sp, -1, 4)\n        yp = jnp.roll(sp, -1, 3)\n        zp = jnp.roll(sp, -1, 2).at[:, :, -1].set(0.0)\n        E = -c4(par, 0) * jnp.sum(sp * (xp + yp + zp), axis=1)\n        sx, sy, sz = sp[:, 0], sp[:, 1], sp[:, 2]\n        E = E + (k1 * ((sx * sy) ** 2 + (sx * sz) ** 2 + (sy * sz) ** 2)\n                 + c4(par, 1) * (sx * sy * sz) ** 2)\n        E = E - c4(par, 3) * (sy * xp[:, 2] - sz * xp[:, 1]\n                              + sz * yp[:, 0] - sx * yp[:, 2])\n        E = E - c4(par, 2) * (sx * par[:, 4].reshape(-1, 1, 1, 1)\n                              + sz * par[:, 5].reshape(-1, 1, 1, 1))\n        return jnp.sum(E * disco[None], axis=(-3, -2, -1))\n\n    def mag(sp):\n        M = jnp.sum(sp * disco[None, None], axis=(2, 3, 4)) / n_act   # (B, 3)\n        return jnp.linalg.norm(M, axis=1), M[:, 2]\n\n    def _omega(a, b, c):\n        num = jnp.sum(a * jnp.cross(b, c, axis=1), axis=1)\n        den = (1.0 + jnp.sum(a * b, 1) + jnp.sum(b * c, 1)\n               + jnp.sum(c * a, 1))\n        return 2.0 * jnp.arctan2(num, den)\n\n    valid = (disk2d[:-1, :-1] & disk2d[:-1, 1:]\n             & disk2d[1:, :-1] & disk2d[1:, 1:])\n\n    def topo(sp):\n        """Berg-Luscher charge per layer: quantised, unlike a difference."""\n        s00 = sp[:, :, :, :-1, :-1]; s10 = sp[:, :, :, :-1, 1:]\n        s01 = sp[:, :, :, 1:, :-1];  s11 = sp[:, :, :, 1:, 1:]\n        w = _omega(s00, s10, s11) + _omega(s00, s11, s01)\n        return jnp.sum(jnp.where(valid, w, 0.0), axis=(-2, -1)) / (\n            4.0 * jnp.pi)\n\n    return dict(energy=energy, mag=mag, topo=topo, n_active=n_act)\n\n\ndef build_anneal(geo, BY=4, num_warps=4, So=1.0):\n    """The full temperature scan, with the Triton substep inside.\n\n    Structurally identical to `sim_core.build_engine`\'s `anneal`: a `lax.scan`\n    over betas, a thermalisation `fori_loop`, then a measurement `fori_loop`\n    accumulating the SAME eight quantities in the SAME order, so the result\n    feeds `sim_core.reduce_obs` unchanged.\n\n    The one difference is the random stream. `sim_core` splits a Threefry key\n    down the loop; the kernel is counter-based, so a step counter rides the\n    carry and every substep gets its own Philox offset. That makes the chain\n    reproducible from `seed` without carrying key state through the scan.\n    """\n    import jax\n    import jax.numpy as jnp\n\n    sub = build(geo, BY=BY, num_warps=num_warps, So=So)\n    O = build_obs(geo)\n    LZ = int(geo["L"])\n\n    def anneal(sp, seed, betas, par, k1, n_therm, n_meas):\n        n_therm, n_meas = int(n_therm), int(n_meas)\n\n        def pair(s, c, bt):\n            s = sub(s, k1, par, bt, c, 0)\n            return sub(s, k1, par, bt, c + 1, 1), c + 2\n\n        def per_T(carry, bt):\n            s, c = carry\n\n            def therm(_, v):\n                s, c = v\n                return pair(s, c, bt)\n            s, c = jax.lax.fori_loop(0, n_therm, therm, (s, c))\n\n            # Fluctuations are taken about a reference fixed once per\n            # temperature. Accumulating raw E and E^2 in float32 destroys the\n            # variance by cancellation -- one ulp of <E^2> can exceed the\n            # variance itself, and Cv comes out negative. A constant shift\n            # leaves the variance unchanged, so this is exact.\n            E_ref = O["energy"](s, par, k1)\n            M_ref, _ = O["mag"](s)\n            z0 = jnp.zeros(s.shape[0], jnp.float32)\n\n            def meas(_, v):\n                s, c, sE, sE2, sM, sM2, sMz, sQ = v\n                s, c = pair(s, c, bt)\n                dE = O["energy"](s, par, k1) - E_ref\n                M, Mz = O["mag"](s)\n                dM = M - M_ref\n                Q = O["topo"](s)[:, LZ // 2]\n                return (s, c, sE + dE, sE2 + dE * dE, sM + dM,\n                        sM2 + dM * dM, sMz + Mz, sQ + Q)\n\n            s, c, sE, sE2, sM, sM2, sMz, sQ = jax.lax.fori_loop(\n                0, n_meas, meas, (s, c, z0, z0, z0, z0, z0, z0))\n            out = jnp.stack([sE / n_meas, sE2 / n_meas,\n                             sM / n_meas, sM2 / n_meas,\n                             sMz / n_meas, sQ / n_meas, E_ref, M_ref])\n            # Replica 0, middle layer, z component -- the same snapshot\n            # `sim_core` takes, read off the planar layout.\n            snap = s[0, 2, LZ // 2]\n            return (s, c), (out, snap)\n\n        # `int(seed)` forced concreteness, which is fine under `jax.jit`\n        # with a Python int closed over but fails under `pmap`, where the\n        # per-device seed arrives as a tracer. `asarray` accepts both.\n        c0 = jnp.asarray(seed, jnp.int32) * jnp.int32(1000003)\n        (s, _), (all_obs, snaps) = jax.lax.scan(per_T, (sp, c0), betas)\n        return all_obs, s, snaps\n\n    anneal.meta = dict(sub.meta)\n    return anneal\n\n\n# ------------------------------------------- the observables, also in Triton\n#\n# The ladder measurement put a number on what the substep could NOT reach:\n# the bare substep ran 66.6% faster, the full ladder only 59.3%, and those\n# seven points are the measurement loop, which recomputes the energy, the\n# magnetisation and the topological charge in JAX on every measured sweep.\n# These two kernels are that ceiling.\n#\n# Both emit PER-BLOCK PARTIALS rather than accumulating with atomics. Atomic\n# float adds would make the result depend on block completion order, so two\n# runs of the same state would disagree in the last bits and the exactness\n# test below would have nothing to stand on. The partial array is tiny\n# (B x LZ*n_strip x 4), so the final sum costs nothing in JAX.\n\n@triton.jit\ndef _eM_kernel(\n    sp_ptr,          # (B, 3, LZ, N, N)  f32\n    k1_ptr,          # (B, LZ, N, N)     f32\n    par_ptr,         # (B, 6)            f32\n    disk_ptr,        # (N, N)            f32\n    out_ptr,         # (B, LZ*n_strip, 4) f32  -> E, Mx, My, Mz\n    N: tl.constexpr,\n    LZ: tl.constexpr,\n    BY: tl.constexpr,\n    BX: tl.constexpr,\n    NSTRIP: tl.constexpr,\n    SO: tl.constexpr,\n):\n    b = tl.program_id(0)\n    z = tl.program_id(1)\n    ty = tl.program_id(2)\n\n    NN = N * N\n    sb = 3 * LZ * NN\n    sc = LZ * NN\n\n    ys = ty * BY + tl.arange(0, BY)\n    xs = tl.arange(0, BX)\n    live = (ys < N)[:, None] & (xs < N)[None, :]\n    yc = tl.where(ys < N, ys, 0)\n    xc = tl.where(xs < N, xs, 0)\n\n    # Each bond counted once: the +x, +y and +z shifts only.\n    ypi = tl.where(yc + 1 >= N, 0, yc + 1)\n    xpi = tl.where(xc + 1 >= N, 0, xc + 1)\n    o_c = yc[:, None] * N + xc[None, :]\n    o_yp = ypi[:, None] * N + xc[None, :]\n    o_xp = yc[:, None] * N + xpi[None, :]\n    zp = tl.minimum(z + 1, LZ - 1)\n    gp = tl.where(z < LZ - 1, 1.0, 0.0)\n\n    msk = live & (tl.load(disk_ptr + o_c, mask=live, other=0.0) > 0.5)\n    w = tl.where(msk, 1.0, 0.0)\n\n    Jex = tl.load(par_ptr + b * 6 + 0)\n    Kan2 = tl.load(par_ptr + b * 6 + 1)\n    Hex = tl.load(par_ptr + b * 6 + 2)\n    KDM = tl.load(par_ptr + b * 6 + 3)\n    cosg = tl.load(par_ptr + b * 6 + 4)\n    sing = tl.load(par_ptr + b * 6 + 5)\n\n    sca = sp_ptr + b * sb\n    k1 = tl.load(k1_ptr + b * LZ * NN + z * NN + o_c, mask=live, other=0.0)\n\n    p0 = sca + 0 * sc; p1 = sca + 1 * sc; p2 = sca + 2 * sc\n    s0 = tl.load(p0 + z * NN + o_c, mask=live, other=0.0)\n    s1 = tl.load(p1 + z * NN + o_c, mask=live, other=0.0)\n    s2 = tl.load(p2 + z * NN + o_c, mask=live, other=0.0)\n    x0 = tl.load(p0 + z * NN + o_xp, mask=live, other=0.0)\n    x1 = tl.load(p1 + z * NN + o_xp, mask=live, other=0.0)\n    x2 = tl.load(p2 + z * NN + o_xp, mask=live, other=0.0)\n    y0 = tl.load(p0 + z * NN + o_yp, mask=live, other=0.0)\n    y1 = tl.load(p1 + z * NN + o_yp, mask=live, other=0.0)\n    y2 = tl.load(p2 + z * NN + o_yp, mask=live, other=0.0)\n    z0 = gp * tl.load(p0 + zp * NN + o_c, mask=live, other=0.0)\n    z1 = gp * tl.load(p1 + zp * NN + o_c, mask=live, other=0.0)\n    z2 = gp * tl.load(p2 + zp * NN + o_c, mask=live, other=0.0)\n\n    E = -Jex * (s0 * (x0 + y0 + z0) + s1 * (x1 + y1 + z1)\n                + s2 * (x2 + y2 + z2))\n    i4 = 1.0 / (SO * SO * SO * SO)\n    i6 = i4 / (SO * SO)\n    E += (k1 * i4 * ((s0 * s1) * (s0 * s1) + (s0 * s2) * (s0 * s2)\n                     + (s1 * s2) * (s1 * s2))\n          + Kan2 * i6 * (s0 * s1 * s2) * (s0 * s1 * s2))\n    E -= KDM * (s1 * x2 - s2 * x1 + s2 * y0 - s0 * y2)\n    E -= Hex * (s0 * cosg + s2 * sing)\n\n    ob = out_ptr + b * (LZ * NSTRIP) * 4 + (z * NSTRIP + ty) * 4\n    tl.store(ob + 0, tl.sum(E * w))\n    tl.store(ob + 1, tl.sum(s0 * w))\n    tl.store(ob + 2, tl.sum(s1 * w))\n    tl.store(ob + 3, tl.sum(s2 * w))\n\n\n@triton.jit\ndef _topo_kernel(\n    sp_ptr,          # (B, 3, LZ, N, N)  f32\n    out_ptr,         # (B, 4, N-1, N-1)  f32  -> n1, d1, n2, d2\n    N: tl.constexpr,\n    LZ: tl.constexpr,\n    ZMID: tl.constexpr,\n    BY: tl.constexpr,\n    BX: tl.constexpr,\n):\n    """The plaquette terms of the Berg-Luscher charge, middle layer only.\n\n    It emits the four arguments of the two `atan2` rather than the angle:\n    Triton 3.8 has no `atan2` outside libdevice, and the branch matters here\n    because the denominator `1 + a.b + b.c + c.a` can go negative on a\n    frustrated plaquette, so `atan(n/d)` is not a substitute. The twelve\n    loads and the cross products -- the expensive part -- stay in the kernel\n    and JAX finishes with one cheap elementwise pass over 38x38.\n\n    One z plane, because that is the only layer `anneal` reads, so the\n    plaquettes are not evaluated five times over.\n    """\n    b = tl.program_id(0)\n    ty = tl.program_id(1)\n    tx = tl.program_id(2)\n\n    NN = N * N\n    sb = 3 * LZ * NN\n    sc = LZ * NN\n    M = N - 1                      # the plaquette grid\n\n    ys = ty * BY + tl.arange(0, BY)\n    xs = tx * BX + tl.arange(0, BX)\n    # Spans y..y+1 and x..x+1 with NO wrap, matching `spins[..., :-1, :-1, :]`.\n    live = (ys < M)[:, None] & (xs < M)[None, :]\n    yc = tl.where(ys < M, ys, 0)\n    xc = tl.where(xs < M, xs, 0)\n\n    o00 = yc[:, None] * N + xc[None, :]\n    o10 = yc[:, None] * N + (xc + 1)[None, :]\n    o01 = (yc + 1)[:, None] * N + xc[None, :]\n    o11 = (yc + 1)[:, None] * N + (xc + 1)[None, :]\n\n    base = sp_ptr + b * sb + ZMID * NN\n    a0 = tl.load(base + 0 * sc + o00, mask=live, other=0.0)\n    a1 = tl.load(base + 1 * sc + o00, mask=live, other=0.0)\n    a2 = tl.load(base + 2 * sc + o00, mask=live, other=0.0)\n    u0 = tl.load(base + 0 * sc + o10, mask=live, other=0.0)\n    u1 = tl.load(base + 1 * sc + o10, mask=live, other=0.0)\n    u2 = tl.load(base + 2 * sc + o10, mask=live, other=0.0)\n    c0 = tl.load(base + 0 * sc + o01, mask=live, other=0.0)\n    c1 = tl.load(base + 1 * sc + o01, mask=live, other=0.0)\n    c2 = tl.load(base + 2 * sc + o01, mask=live, other=0.0)\n    e0 = tl.load(base + 0 * sc + o11, mask=live, other=0.0)\n    e1 = tl.load(base + 1 * sc + o11, mask=live, other=0.0)\n    e2 = tl.load(base + 2 * sc + o11, mask=live, other=0.0)\n\n    # omega(a, u, v) = 2 atan2( a . (u x v), 1 + a.u + u.v + v.a )\n    n1 = (a0 * (u1 * e2 - u2 * e1) + a1 * (u2 * e0 - u0 * e2)\n          + a2 * (u0 * e1 - u1 * e0))\n    d1 = (1.0 + (a0 * u0 + a1 * u1 + a2 * u2)\n          + (u0 * e0 + u1 * e1 + u2 * e2)\n          + (e0 * a0 + e1 * a1 + e2 * a2))\n    n2 = (a0 * (e1 * c2 - e2 * c1) + a1 * (e2 * c0 - e0 * c2)\n          + a2 * (e0 * c1 - e1 * c0))\n    d2 = (1.0 + (a0 * e0 + a1 * e1 + a2 * e2)\n          + (e0 * c0 + e1 * c1 + e2 * c2)\n          + (c0 * a0 + c1 * a1 + c2 * a2))\n\n    off = yc[:, None] * M + xc[None, :]\n    ob = out_ptr + b * 4 * M * M\n    tl.store(ob + 0 * M * M + off, n1, mask=live)\n    tl.store(ob + 1 * M * M + off, d1, mask=live)\n    tl.store(ob + 2 * M * M + off, n2, mask=live)\n    tl.store(ob + 3 * M * M + off, d2, mask=live)\n\n\ndef build_obs_triton(geo, BY=4, BX=None, num_warps=4, So=1.0):\n    """E, M and Q from two Triton kernels plus one tiny JAX reduction."""\n    import jax\n    import jax.numpy as jnp\n    import jax_triton as jt\n\n    N, LZ = int(geo["N"]), int(geo["L"])\n    BX = int(BX) if BX else 1 << (N - 1).bit_length()\n    BY = int(BY)\n    nst = -(-N // BY)\n    M = N - 1\n    nqy, nqx = -(-M // BY), -(-M // BX)\n    disk = jnp.asarray(geo["disk2d"].astype(np.float32))\n    d2d = geo["disk2d"]\n    qvalid = jnp.asarray((d2d[:-1, :-1] & d2d[:-1, 1:]\n                          & d2d[1:, :-1] & d2d[1:, 1:]))\n    n_act = float(geo["n_active"])\n    four_pi = 4.0 * float(np.pi)\n\n    def eM(sp, par, k1):\n        B = sp.shape[0]\n        part = jt.triton_call(\n            sp, k1, par, disk,\n            kernel=_eM_kernel,\n            out_shape=jax.ShapeDtypeStruct((B, LZ * nst, 4), jnp.float32),\n            grid=(B, LZ, nst),\n            num_warps=int(num_warps),\n            N=N, LZ=LZ, BY=BY, BX=BX, NSTRIP=nst, SO=float(So))\n        tot = jnp.sum(part, axis=1)\n        M = tot[:, 1:4] / n_act\n        # the whole vector comes back too: the kernel already computed\n        # Mx and My, and only this line was discarding them.\n        return tot[:, 0], jnp.linalg.norm(M, axis=1), M[:, 2], M\n\n    def topo_mid(sp):\n        B = sp.shape[0]\n        t = jt.triton_call(\n            sp,\n            kernel=_topo_kernel,\n            out_shape=jax.ShapeDtypeStruct((B, 4, M, M), jnp.float32),\n            grid=(B, nqy, nqx),\n            num_warps=int(num_warps),\n            N=N, LZ=LZ, ZMID=LZ // 2, BY=BY, BX=BX)\n        w = 2.0 * (jnp.arctan2(t[:, 0], t[:, 1])\n                   + jnp.arctan2(t[:, 2], t[:, 3]))\n        return jnp.sum(jnp.where(qvalid, w, 0.0), axis=(1, 2)) / four_pi\n\n    return dict(eM=eM, topo_mid=topo_mid, n_active=n_act,\n                meta=dict(BY=BY, BX=BX, n_strip=nst, q_grid=(nqy, nqx)))\n\n\ndef build_anneal_full(geo, BY=4, num_warps=4, So=1.0, with_mvec=False):\n    """`with_mvec` adds a fourth return: the mean magnetisation VECTOR per\n    temperature and replica, shape (n_T, B, 3).\n\n    It is opt-in because ten call sites unpack three values, and the vector is\n    not free -- it is one more accumulator carried through the measurement\n    loop. What it buys is a PER-REPLICA record: everything else this function\n    returns has been averaged over replicas by the time it is read."""\n    """The anneal with BOTH the substep and the observables in Triton."""\n    import jax\n    import jax.numpy as jnp\n\n    sub = build(geo, BY=BY, num_warps=num_warps, So=So)\n    O = build_obs_triton(geo, BY=BY, num_warps=num_warps, So=So)\n    LZ = int(geo["L"])\n\n    def anneal(sp, seed, betas, par, k1, n_therm, n_meas):\n        n_therm, n_meas = int(n_therm), int(n_meas)\n\n        def pair(s, c, bt):\n            s = sub(s, k1, par, bt, c, 0)\n            return sub(s, k1, par, bt, c + 1, 1), c + 2\n\n        def per_T(carry, bt):\n            s, c = carry\n\n            def therm(_, v):\n                s, c = v\n                return pair(s, c, bt)\n            s, c = jax.lax.fori_loop(0, n_therm, therm, (s, c))\n\n            E_ref, M_ref, _, _ = O["eM"](s, par, k1)\n            z0 = jnp.zeros(s.shape[0], jnp.float32)\n\n            zv = jnp.zeros((s.shape[0], 3), jnp.float32)\n\n            def meas(_, v):\n                s, c, sE, sE2, sM, sM2, sMz, sQ, sMv = v\n                s, c = pair(s, c, bt)\n                E, M, Mz, Mv = O["eM"](s, par, k1)\n                dE = E - E_ref\n                dM = M - M_ref\n                Q = O["topo_mid"](s)\n                return (s, c, sE + dE, sE2 + dE * dE, sM + dM,\n                        sM2 + dM * dM, sMz + Mz, sQ + Q, sMv + Mv)\n\n            s, c, sE, sE2, sM, sM2, sMz, sQ, sMv = jax.lax.fori_loop(\n                0, n_meas, meas, (s, c, z0, z0, z0, z0, z0, z0, zv))\n            out = jnp.stack([sE / n_meas, sE2 / n_meas,\n                             sM / n_meas, sM2 / n_meas,\n                             sMz / n_meas, sQ / n_meas, E_ref, M_ref])\n            if with_mvec:\n                return (s, c), (out, s[0, 2, LZ // 2], sMv / n_meas)\n            return (s, c), (out, s[0, 2, LZ // 2])\n\n        # `int(seed)` forced concreteness, which is fine under `jax.jit`\n        # with a Python int closed over but fails under `pmap`, where the\n        # per-device seed arrives as a tracer. `asarray` accepts both.\n        c0 = jnp.asarray(seed, jnp.int32) * jnp.int32(1000003)\n        if with_mvec:\n            (s, _), (all_obs, snaps, mvec) = jax.lax.scan(\n                per_T, (sp, c0), betas)\n            return all_obs, s, snaps, mvec\n        (s, _), (all_obs, snaps) = jax.lax.scan(per_T, (sp, c0), betas)\n        return all_obs, s, snaps\n\n    anneal.meta = dict(sub.meta, **O["meta"])\n    return anneal\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "f397f10c5870a42bfe273a101f5bbada5a79ad6259d140cf636ee51584ca5a57", "triton_halo.py alterado"
open("triton_halo.py", "w").write(_src)
print("triton_halo.py  {} bytes  sha256 ok".format(len(_src)))


In [ ]:
# sim_multishell.py — idéntico byte a byte al del repositorio. sha256 f7416703e62f529f4a75d3a929d7dc3fb6f76dd6510434a728f5c40a95948d16
import hashlib
_src = '"""The Monte Carlo engine generalised to the first four exchange shells.\n\n`sim_core` carries first-shell exchange only, and for that case a two-colour\ncheckerboard is an EXACT reordering of the sequential sweep: every first\nneighbour sits at an odd displacement, so it always has the opposite parity of\n``(x+y+z)``. Turning on a shell whose displacements sum to an even number breaks\nthat, and the fix is not to forbid the shell -- it is to use more sublattices.\n\nThe minimal colouring is decided by which shells are active, and it is VERIFIED\nhere rather than asserted (see `check_colouring`):\n\n    active shells        colouring                     sublattices\n    {1}, {1,3}           (x+y+z) mod 2                      2\n    anything with 2      (x mod 2, y mod 2, z mod 2)        8\n    anything with 4      (x mod 3, y mod 3, z mod 3)       27\n\nShells 1 and 3 have odd displacement sums and keep the graph bipartite; shells\n2 and 4 do not. Shell 4 reaches distance 2 along an axis, which no stride-2\ncolouring can separate, so it needs stride 3 on all three axes.\n\nThe cost of a wider colouring is immediate and worth stating: a sweep becomes\none pass per colour, and each pass updates only 1/n of the sites. Going from\ntwo sublattices to twenty-seven does not make a sweep thirteen times slower --\nthe work per sweep is the same -- but it does cut the parallelism available in\neach pass by that factor, which on a GPU is what the cost follows.\n\nThe DMI term stays on first-shell in-plane bonds whatever else is switched on:\nit is interfacial by construction, not a generic chiral coupling.\n"""\n\nfrom __future__ import annotations\n\nimport itertools\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\nfrom jax import random\n\n# The displacement vectors of each shell on a simple cubic lattice, by\n# squared distance: 1, 2, 3 and 4.\n_SHELL = {\n    1: [d for d in itertools.product((-1, 0, 1), repeat=3)\n        if sum(x * x for x in d) == 1],\n    2: [d for d in itertools.product((-1, 0, 1), repeat=3)\n        if sum(x * x for x in d) == 2],\n    3: [d for d in itertools.product((-1, 0, 1), repeat=3)\n        if sum(x * x for x in d) == 3],\n    4: [d for d in itertools.product((-2, 0, 2), repeat=3)\n        if sum(x * x for x in d) == 4],\n}\n\n_COLOURINGS = [\n    ("tablero", 2, lambda x, y, z: (x + y + z) % 2),\n    ("paridad por eje", 8, lambda x, y, z: (x % 2) * 4 + (y % 2) * 2 + z % 2),\n    ("paso 3 por eje", 27, lambda x, y, z: (x % 3) * 9 + (y % 3) * 3 + z % 3),\n]\n\n\ndef shells_of(Jex2=0.0, Jex3=0.0, Jex4=0.0):\n    """Which shells are active. The first is always on and normalised to 1."""\n    out = [1]\n    for k, v in ((2, Jex2), (3, Jex3), (4, Jex4)):\n        if abs(float(v)) > 0.0:\n            out.append(k)\n    return out\n\n\ndef check_colouring(colour, shells, extent=6):\n    """True when no displacement of any active shell preserves the colour.\n\n    Checked over a window of the lattice rather than argued from parity, so a\n    colouring that happens to fail somewhere is caught here and not in the\n    physics.\n    """\n    for k in shells:\n        for d in _SHELL[k]:\n            for p in itertools.product(range(extent), repeat=3):\n                q = tuple(p[i] + d[i] for i in range(3))\n                if colour(*p) == colour(*q):\n                    return False\n    return True\n\n\ndef pick_colouring(shells):\n    """The narrowest verified colouring for those shells."""\n    for name, n, f in _COLOURINGS:\n        if check_colouring(f, shells):\n            return name, n, f\n    raise ValueError(f"ninguno de los coloreos conocidos sirve para {shells}")\n\n\ndef build_geometry(Rd: float = 18.30, L: int = 5, shells=(1,)):\n    """Disc mask, the per-shell neighbour offsets, and the sublattice masks."""\n    shells = tuple(sorted(set(int(s) for s in shells)))\n    L2 = int(Rd) + 1\n    N = 2 * L2 + 1\n    x = np.arange(-L2, L2 + 1)\n    xg, yg = np.meshgrid(x, x, indexing="xy")\n    disk2d = (xg ** 2 + yg ** 2) <= Rd ** 2\n    disco = np.broadcast_to(disk2d, (L, N, N)).copy()\n\n    name, ncol, f = pick_colouring(shells)\n    zc, yc, xc = np.indices((L, N, N))\n    colour = f(xc, yc, zc)\n    masks = [((colour == c) & disco) for c in range(ncol)]\n    # A colour class empty inside the disc would be a wasted pass.\n    masks = [m for m in masks if m.any()]\n\n    # Surface = fewer than six in-disc FIRST neighbours, as in `sim_core`.\n    surface = np.zeros_like(disco, dtype=bool)\n    for z in range(L):\n        for yy in range(N):\n            for xx in range(N):\n                if not disco[z, yy, xx]:\n                    continue\n                n = 0\n                for dz, dy, dx in _SHELL[1]:\n                    nz, ny, nx = z + dz, (yy + dy) % N, (xx + dx) % N\n                    if 0 <= nz < L and disco[nz, ny, nx]:\n                        n += 1\n                if n < 6:\n                    surface[z, yy, xx] = True\n\n    return dict(L=L, N=N, Rd=Rd, disco=disco, surface=surface, disk2d=disk2d,\n                shells=shells, colouring=name, n_sublattices=len(masks),\n                masks=masks, n_active=int(disco.sum()),\n                n_surface=int(surface.sum()))\n\n\ndef _shift(spins, d, L):\n    """One neighbour translate. x and y wrap; z is open, so it is zeroed."""\n    dz, dy, dx = d\n    out = spins\n    if dx:\n        out = jnp.roll(out, -dx, 3)\n    if dy:\n        out = jnp.roll(out, -dy, 2)\n    if dz:\n        out = jnp.roll(out, -dz, 1)\n        # the |dz| layers that rolled in from the other end have no neighbour\n        if dz > 0:\n            out = out.at[:, L - dz:].set(0.0)\n        else:\n            out = out.at[:, :(-dz)].set(0.0)\n    return out\n\n\ndef build_engine(geo, So=1.0):\n    """Returns (substep, energy, magnetization, topo_charge, anneal).\n\n    `substep` takes the index of the sublattice to update rather than a mask,\n    so the caller loops over `geo["n_sublattices"]` instead of over two\n    hardcoded colours.\n    """\n    L, N = geo["L"], geo["N"]\n    shells = geo["shells"]\n    disco_j = jnp.asarray(geo["disco"][None, ..., None])\n    surf_j = jnp.asarray(geo["surface"][None, ...])\n    disk2d_j = jnp.asarray(geo["disk2d"])\n    masks = [jnp.asarray(m[None, ..., None]) for m in geo["masks"]]\n    n_act = geo["n_active"]\n\n    def _an(s, Kan2, K1_eff):\n        sx, sy, sz = s[..., 0], s[..., 1], s[..., 2]\n        e = (K1_eff / So ** 4) * ((sx * sy) ** 2 + (sx * sz) ** 2\n                                  + (sy * sz) ** 2)\n        return e + (Kan2 / So ** 6) * (sx * sy * sz) ** 2\n\n    def _coupling(p, k):\n        """The exchange constant of shell k. The first is normalised to 1."""\n        return p["Jex"] if k == 1 else p[f"Jex{k}"]\n\n    def substep(spins, key, beta, sub, p, K1_eff):\n        mask = masks[sub]\n        # the weighted neighbour sum over every active shell\n        sv = 0.0\n        for k in shells:\n            acc = 0.0\n            for d in _SHELL[k]:\n                acc = acc + _shift(spins, d, L)\n            # the coupling columns are (B,1,1,1) and `acc` carries the\n            # component axis, so the coupling needs a trailing axis to broadcast\n            sv = sv + _coupling(p, k)[..., None] * acc\n\n        kt, ka = random.split(key)\n        tr = random.normal(kt, spins.shape)\n        tr = tr / jnp.linalg.norm(tr, axis=-1, keepdims=True) * So\n        tr = jnp.where(mask, tr, spins)\n        dS = tr - spins\n\n        dE = -jnp.sum(dS * sv, axis=-1)\n        dE = dE + _an(tr, p["Kan2"], K1_eff) - _an(spins, p["Kan2"], K1_eff)\n        # Interfacial DMI: first-shell in-plane bonds only, whatever else is on.\n        xp = _shift(spins, (0, 0, 1), L); xm = _shift(spins, (0, 0, -1), L)\n        yp = _shift(spins, (0, 1, 0), L); ym = _shift(spins, (0, -1, 0), L)\n        dE = dE - p["KDM"] * (dS[..., 1] * (xp[..., 2] - xm[..., 2])\n                              - dS[..., 2] * (xp[..., 1] - xm[..., 1])\n                              + dS[..., 2] * (yp[..., 0] - ym[..., 0])\n                              - dS[..., 0] * (yp[..., 2] - ym[..., 2]))\n        dE = dE - p["Hex"] * (dS[..., 0] * jnp.cos(p["g"])\n                              + dS[..., 2] * jnp.sin(p["g"]))\n\n        r = random.uniform(ka, dE.shape)\n        acc = ((dE < 0) | (r < jnp.exp(-dE * beta)))[..., None] & mask\n        return jnp.where(acc, tr, spins)\n\n    def energy(spins, p, K1_eff):\n        """Total energy, each bond counted once: only the positive half of each\n        shell\'s displacement set is summed."""\n        E = 0.0\n        for k in shells:\n            half = [d for d in _SHELL[k] if d > (0, 0, 0)]\n            acc = 0.0\n            for d in half:\n                acc = acc + _shift(spins, d, L)\n            E = E - _coupling(p, k) * jnp.sum(spins * acc, axis=-1)\n        sx, sy, sz = spins[..., 0], spins[..., 1], spins[..., 2]\n        E = E + _an(spins, p["Kan2"], K1_eff)\n        xp = _shift(spins, (0, 0, 1), L); yp = _shift(spins, (0, 1, 0), L)\n        E = E - p["KDM"] * (sy * xp[..., 2] - sz * xp[..., 1]\n                            + sz * yp[..., 0] - sx * yp[..., 2])\n        E = E - p["Hex"] * (sx * jnp.cos(p["g"]) + sz * jnp.sin(p["g"]))\n        return jnp.sum(E * disco_j[..., 0], axis=(-3, -2, -1))\n\n    def magnetization(spins):\n        M = jnp.sum(spins * disco_j, axis=(1, 2, 3)) / n_act\n        return jnp.linalg.norm(M, axis=-1), M[:, 2]\n\n    def _omega(s1, s2, s3):\n        num = jnp.sum(s1 * jnp.cross(s2, s3), axis=-1)\n        den = (1.0 + jnp.sum(s1 * s2, -1) + jnp.sum(s2 * s3, -1)\n               + jnp.sum(s3 * s1, -1))\n        return 2.0 * jnp.arctan2(num, den)\n\n    def topo_charge(spins):\n        s00 = spins[..., :-1, :-1, :]; s10 = spins[..., :-1, 1:, :]\n        s01 = spins[..., 1:, :-1, :];  s11 = spins[..., 1:, 1:, :]\n        valid = (disk2d_j[:-1, :-1] & disk2d_j[:-1, 1:]\n                 & disk2d_j[1:, :-1] & disk2d_j[1:, 1:])\n        w = _omega(s00, s10, s11) + _omega(s00, s11, s01)\n        return jnp.sum(jnp.where(valid, w, 0.0), axis=(-2, -1)) / (4.0 * jnp.pi)\n\n    def anneal(spins, key, betas, p, n_therm, n_meas):\n        K1_eff = jnp.where(surf_j, p["KanS"], p["Kan1"])\n        nsub = len(masks)\n\n        def sweep(s, k, beta):\n            """One sweep is one pass per sublattice, in order."""\n            for c in range(nsub):\n                k, kc = random.split(k)\n                s = substep(s, kc, beta, c, p, K1_eff)\n            return s, k\n\n        def per_T(carry, beta):\n            s, k = carry\n\n            def therm(_, v):\n                return sweep(v[0], v[1], beta)\n            s, k = jax.lax.fori_loop(0, n_therm, therm, (s, k))\n\n            b = s.shape[0]\n            z0 = jnp.zeros(b)\n            E_ref = energy(s, p, K1_eff)\n            M_ref, _ = magnetization(s)\n\n            def meas(_, v):\n                s, k, sE, sE2, sM, sM2, sMz, sQ = v\n                s, k = sweep(s, k, beta)\n                dE = energy(s, p, K1_eff) - E_ref\n                M, Mz = magnetization(s)\n                dM = M - M_ref\n                Q = topo_charge(s)[:, L // 2]\n                return (s, k, sE + dE, sE2 + dE * dE, sM + dM,\n                        sM2 + dM * dM, sMz + Mz, sQ + Q)\n\n            s, k, sE, sE2, sM, sM2, sMz, sQ = jax.lax.fori_loop(\n                0, n_meas, meas, (s, k, z0, z0, z0, z0, z0, z0))\n            obs = jnp.stack([sE / n_meas, sE2 / n_meas, sM / n_meas,\n                             sM2 / n_meas, sMz / n_meas, sQ / n_meas,\n                             E_ref, M_ref])\n            return (s, k), (obs, s[0, L // 2, :, :, 2])\n\n        (s, _), (all_obs, snaps) = jax.lax.scan(per_T, (spins, key), betas)\n        return all_obs, s, snaps\n\n    return dict(substep=substep, energy=energy, magnetization=magnetization,\n                topo_charge=topo_charge, anneal=anneal,\n                n_sublattices=len(masks))\n\n\ndef init_spins(key, geo, batch, So=1.0):\n    L, N = geo["L"], geo["N"]\n    x = random.normal(key, (batch, L, N, N, 3))\n    x = x / jnp.linalg.norm(x, axis=-1, keepdims=True) * So\n    return x * jnp.asarray(geo["disco"][None, ..., None])\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "f7416703e62f529f4a75d3a929d7dc3fb6f76dd6510434a728f5c40a95948d16", "sim_multishell.py alterado"
open("sim_multishell.py", "w").write(_src)
print("sim_multishell.py  {} bytes  sha256 ok".format(len(_src)))


In [ ]:
# triton_bench.py — idéntico byte a byte al del repositorio. sha256 0c115e3ce8af2a74ac7a1da682be3d02bae4e44c4d4374238e8b8b66e764d003
import hashlib
_src = '"""Does the hand-written kernel travel? One self-contained benchmark.\n\nThe whole Triton result was measured on H100s rented by the hour. That leaves\ntwo things unknown. Whether the kernel even COMPILES elsewhere -- Triton wants\ncompute capability 7.0 or better, so Kaggle\'s P100 (6.0) should refuse it\nwhile its T4 (7.5) should not. And whether the gain is an H100 property: the\nH100 has 132 SMs and enormous L2, and a stencil that lives in registers might\nbehave very differently on a 40-SM T4.\n\nSo this module runs, in order:\n\n  1. the environment, including compute capability, because a refusal there is\n     a result and not a failure;\n  2. exactness -- the stencil against `sim_core.substep` on IDENTICAL\n     randomness, which must give zero differing spins;\n  3. physics -- both engines to equilibrium with their own randomness, whose\n     energies must agree within the spread the run itself reports;\n  4. the substep rate, production against the kernel, over a grid of layer\n     counts and batches, which also scores the width rule SEPARATELY for each\n     engine;\n  5. a short ladder, so the figure quoted is the full anneal and not a step.\n\nNothing is timed before the physics agrees, and every arm is interleaved so a\nGPU that warms up spreads its drift across both.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport time\n\nimport numpy as np\n\n\ndef _v2(n):\n    return (n & -n).bit_length() - 1\n\n\ndef environment():\n    import subprocess\n    import sys\n    out = {}\n    try:\n        out["smi"] = subprocess.run(\n            ["nvidia-smi", "--query-gpu=name,memory.total,clocks.max.sm,"\n             "driver_version", "--format=csv,noheader"],\n            capture_output=True, text=True).stdout.strip()\n    except Exception as e:\n        out["smi"] = f"nvidia-smi no disponible: {e}"\n    try:\n        import jax\n        out["jax"] = jax.__version__\n        out["devices"] = [str(d) for d in jax.devices()]\n        d = jax.devices()[0]\n        # Compute capability decides whether Triton will emit anything at all.\n        cc = getattr(d, "compute_capability", None)\n        out["compute_capability"] = str(cc) if cc else "desconocida"\n        out["device_kind"] = getattr(d, "device_kind", "?")\n    except Exception as e:\n        out["jax_error"] = f"{type(e).__name__}: {e}"\n    for mod in ("triton", "jax_triton"):\n        try:\n            m = __import__(mod)\n            out[mod] = getattr(m, "__version__", "?")\n        except Exception as e:\n            out[f"{mod}_error"] = f"{type(e).__name__}: {e}"\n    out["python"] = sys.version.split()[0]\n    return out\n\n\ndef run(batches=(64, 65, 100, 101), layers=(5,), repeats=5, steps=200,\n        Rd=18.30, by=4, warps=4, replicas=50, n_temps=20, n_therm=300,\n        n_meas=150, seed=11, out=None, label="kaggle"):\n    import jax\n    import jax.numpy as jnp\n    import sim_core as S\n\n    res = dict(label=label, env=environment(), rows=[], ok=False)\n    print("=== entorno ===", flush=True)\n    for k in ("smi", "device_kind", "compute_capability", "jax", "triton",\n              "jax_triton", "python"):\n        if k in res["env"]:\n            print(f"  {k:<20} {res[\'env\'][k]}", flush=True)\n    cc = res["env"].get("compute_capability", "")\n    if cc and cc.replace(".", "").isdigit() and float(\n            cc if "." in cc else f"{cc[0]}.{cc[1:]}") < 7.0:\n        print(f"\\n  AVISO: capacidad de computo {cc} < 7.0. Triton exige 7.0+,"\n              f" asi que lo mas probable es que se niegue a compilar. Eso es"\n              f" un resultado, no un fallo: elige el acelerador T4 (7.5).",\n              flush=True)\n\n    try:\n        import triton_halo as TH\n    except Exception as e:\n        res["error"] = f"{type(e).__name__}: {e}"\n        print(f"\\nTRITON NO DISPONIBLE: {res[\'error\']}", flush=True)\n        if out:\n            json.dump(res, open(out, "w"), indent=1)\n        return res\n\n    kB = 0.086173404\n    beta = 1.0 / (kB * 5.0)\n    K = int(steps)\n    P0 = dict(Jex=1.0, Kan1=0.177, Kan2=0.0, KanS=0.139, Hex=0.483,\n              KDM=0.880, g=float(np.deg2rad(90.0)))\n\n    def params(b):\n        c = lambda v: jnp.full((b, 1, 1, 1), float(v), jnp.float32)\n        return {k: c(v) for k, v in P0.items()}\n\n    # ---------------------------------------------------- 2. exactness\n    print("\\n=== exactitud: el stencil contra sim_core.substep ===",\n          flush=True)\n    geo = S.build_geometry(Rd=float(Rd), L=int(layers[0]))\n    eng = S.build_engine(geo)\n    surf = jnp.asarray(geo["surface"][None, ...])\n    ev = jnp.asarray(geo["even"][None, ..., None])\n    od = jnp.asarray(geo["odd"][None, ..., None])\n    Bv = 8\n    P = params(Bv)\n    k1 = jnp.where(surf, P["KanS"], P["Kan1"])\n    sp0 = S.init_spins(jax.random.PRNGKey(3), geo, Bv)\n    kt, ka = jax.random.split(jax.random.PRNGKey(101))\n    tr = jax.random.normal(kt, sp0.shape)\n    tr = tr / jnp.linalg.norm(tr, axis=-1, keepdims=True)\n    ru = jax.random.uniform(ka, sp0.shape[:-1])\n    sub = TH.build(geo, BY=int(by), num_warps=int(warps))\n    par = TH.pack_params(P, Bv)\n    spP = TH.planar(sp0)\n    trP = TH.planar(tr)\n    ex = []\n    for parity, mask, nm in ((0, ev, "par"), (1, od, "impar")):\n        # `substep` draws its own randomness, so the reference is rebuilt here\n        # from the SAME tr/ru the kernel is given -- otherwise this compares\n        # two different chains and proves nothing.\n        trm = jnp.where(mask, tr, sp0)\n        dS = trm - sp0\n        xp = jnp.roll(sp0, -1, 3); xm = jnp.roll(sp0, 1, 3)\n        yp = jnp.roll(sp0, -1, 2); ym = jnp.roll(sp0, 1, 2)\n        zp = jnp.roll(sp0, -1, 1).at[:, -1].set(0.0)\n        zm = jnp.roll(sp0, 1, 1).at[:, 0].set(0.0)\n        dE = -P["Jex"] * jnp.sum(dS * (xp + xm + yp + ym + zp + zm), -1)\n        an = lambda s: (k1 * ((s[..., 0] * s[..., 1]) ** 2\n                              + (s[..., 0] * s[..., 2]) ** 2\n                              + (s[..., 1] * s[..., 2]) ** 2)\n                        + P["Kan2"] * (s[..., 0] * s[..., 1] * s[..., 2]) ** 2)\n        dE = dE + an(trm) - an(sp0)\n        dE = dE - P["KDM"] * (dS[..., 1] * (xp[..., 2] - xm[..., 2])\n                              - dS[..., 2] * (xp[..., 1] - xm[..., 1])\n                              + dS[..., 2] * (yp[..., 0] - ym[..., 0])\n                              - dS[..., 0] * (yp[..., 2] - ym[..., 2]))\n        dE = dE - P["Hex"] * (dS[..., 0] * jnp.cos(P["g"])\n                              + dS[..., 2] * jnp.sin(P["g"]))\n        acc = ((dE < 0) | (ru < jnp.exp(-dE * beta)))[..., None] & mask\n        ref = jnp.where(acc, trm, sp0)\n        got, dE_t = sub(spP, k1, par, beta, 0, parity, trP, ru, want_de=True)\n        sel = np.broadcast_to(np.asarray(mask[..., 0]), dE.shape)\n        mad = float(np.max(np.abs(np.asarray(dE)[sel]\n                                  - np.asarray(dE_t)[sel])))\n        bad = int(np.sum(np.asarray(ref) != np.asarray(TH.unplanar(got))))\n        ex.append(dict(sublattice=nm, dE_max_abs=mad, spin_mismatch=bad,\n                       spin_total=int(np.asarray(ref).size)))\n        print(f"  {nm:>5}: dE max |dif| {mad:.3e}   espines distintos "\n              f"{bad}/{np.asarray(ref).size}", flush=True)\n    res["exactness"] = ex\n    res["exact_ok"] = all(r["spin_mismatch"] == 0 for r in ex)\n\n    # ------------------------------------------------------ 3. physics\n    print("\\n=== fisica: las dos cadenas a equilibrio, cada una con su RNG ===",\n          flush=True)\n    Bp = 16\n    Pp = params(Bp)\n    k1p = jnp.where(surf, Pp["KanS"], Pp["Kan1"])\n    parp = TH.pack_params(Pp, Bp)\n    spE = S.init_spins(jax.random.PRNGKey(11), geo, Bp)\n    subp = TH.build(geo, BY=int(by), num_warps=int(warps))\n    n_act = float(geo["n_active"])\n    ph = []\n    for T in (20.0, 5.0, 1.0):\n        bT = 1.0 / (kB * T)\n        n_eq = 400\n\n        def jx(s):\n            def body(i, v):\n                st, k = v\n                k, a, b = jax.random.split(k, 3)\n                st = eng["substep"](st, a, bT, ev, Pp, k1p)\n                st = eng["substep"](st, b, bT, od, Pp, k1p)\n                return st, k\n            return jax.lax.fori_loop(0, n_eq, body,\n                                     (s, jax.random.PRNGKey(77)))[0]\n\n        sj = jax.block_until_ready(jax.jit(jx)(spE))\n        st = TH.unplanar(jax.block_until_ready(jax.jit(\n            TH.sweeps(subp, k1p, parp, bT, n_eq))(TH.planar(spE))))\n        ej = float(jnp.mean(eng["energy"](sj, Pp, k1p)) / n_act)\n        et = float(jnp.mean(eng["energy"](st, Pp, k1p)) / n_act)\n        sd = float(jnp.std(eng["energy"](sj, Pp, k1p)) / n_act)\n        row = dict(T=T, E_prod=ej, E_halo=et, E_sd=sd,\n                   sigma=abs(et - ej) / (sd + 1e-30),\n                   M_prod=float(jnp.mean(eng["magnetization"](sj)[0])),\n                   M_halo=float(jnp.mean(eng["magnetization"](st)[0])))\n        ph.append(row)\n        print(f"  T={T:>5.1f}  E/sitio {ej:+9.5f} / {et:+9.5f}  "\n              f"({row[\'sigma\']:.2f} sigma)   |M| {row[\'M_prod\']:.4f} / "\n              f"{row[\'M_halo\']:.4f}", flush=True)\n    res["physics"] = ph\n    res["physics_ok"] = all(r["sigma"] < 3.0 for r in ph)\n    if not (res["exact_ok"] and res["physics_ok"]):\n        print("\\n  AVISO: la fisica NO coincide. Los tiempos de abajo no son "\n              "comparables.", flush=True)\n\n    # ------------------------------- 4. the substep rate, and the width rule\n    print(f"\\n=== ritmo del paso, {K} barridos por llamada, "\n          f"{repeats} repeticiones intercaladas ===", flush=True)\n    print(f"{\'L\':>3}{\'B\':>6}{\'v2(B)+v2(L)\':>13}{\'produccion\':>13}"\n          f"{\'kernel\':>10}{\'cambio\':>9}", flush=True)\n    for L in layers:\n        g = S.build_geometry(Rd=float(Rd), L=int(L))\n        e2 = S.build_engine(g)\n        s2 = jnp.asarray(g["surface"][None, ...])\n        v2e = jnp.asarray(g["even"][None, ..., None])\n        v2o = jnp.asarray(g["odd"][None, ..., None])\n        for B in batches:\n            Pb = params(B)\n            k1b = jnp.where(s2, Pb["KanS"], Pb["Kan1"])\n            parb = TH.pack_params(Pb, B)\n            spb = S.init_spins(jax.random.PRNGKey(seed + B + L), g, B)\n            spbP = TH.planar(spb)\n            sb = TH.build(g, BY=int(by), num_warps=int(warps))\n\n            def prod(s):\n                def body(i, v):\n                    st, k = v\n                    k, a, b = jax.random.split(k, 3)\n                    st = e2["substep"](st, a, beta, v2e, Pb, k1b)\n                    st = e2["substep"](st, b, beta, v2o, Pb, k1b)\n                    return st, k\n                return jax.lax.fori_loop(0, K, body,\n                                         (s, jax.random.PRNGKey(0)))[0]\n\n            f1 = jax.jit(prod).lower(spb).compile()\n            f2 = jax.jit(TH.sweeps(sb, k1b, parb, beta, K)\n                         ).lower(spbP).compile()\n            jax.block_until_ready(f1(spb)); jax.block_until_ready(f2(spbP))\n            a1, a2 = [], []\n            for _ in range(int(repeats)):\n                t = time.perf_counter(); jax.block_until_ready(f1(spb))\n                a1.append(time.perf_counter() - t)\n                t = time.perf_counter(); jax.block_until_ready(f2(spbP))\n                a2.append(time.perf_counter() - t)\n            m1, m2 = float(np.median(a1)), float(np.median(a2))\n            row = dict(L=int(L), B=int(B), suma=_v2(L) + _v2(B),\n                       prod_us=1e6 * m1 / (K * B),\n                       halo_us=1e6 * m2 / (K * B),\n                       change_pct=100 * (m2 / m1 - 1))\n            res["rows"].append(row)\n            print(f"{L:>3}{B:>6}{row[\'suma\']:>13}{row[\'prod_us\']:>13.3f}"\n                  f"{row[\'halo_us\']:>10.3f}{row[\'change_pct\']:>+8.1f}%",\n                  flush=True)\n\n    pen = {}\n    for L in layers:\n        rs = [r for r in res["rows"] if r["L"] == L]\n        hit = [r for r in rs if r["suma"] >= 2]\n        miss = [r for r in rs if r["suma"] < 2]\n        if hit and miss:\n            for key, lab in (("prod_us", "produccion"), ("halo_us", "kernel")):\n                p = 100 * (float(np.mean([r[key] for r in hit]))\n                           / float(np.mean([r[key] for r in miss])) - 1)\n                pen[f"L{L}_{lab}"] = p\n    if pen:\n        print(f"\\n=== la regla del ancho: castigo por v2(B)+v2(L) >= 2 ===",\n              flush=True)\n        for k, v in pen.items():\n            print(f"  {k:<22}{v:+7.1f}%", flush=True)\n    res["penalty"] = pen\n\n    # ------------------------------------------------- 5. the short ladder\n    print(f"\\n=== escalera corta: {n_temps}T x {n_therm}+{n_meas}, "\n          f"{replicas} replicas ===", flush=True)\n    Bl = int(replicas)\n    Pl = params(Bl)\n    k1l = jnp.where(surf, Pl["KanS"], Pl["Kan1"])\n    parl = TH.pack_params(Pl, Bl)\n    Tl = np.linspace(20.0, 0.1, int(n_temps))\n    btl = jnp.asarray(1.0 / (kB * Tl), jnp.float32)\n    spl = S.init_spins(jax.random.PRNGKey(5), geo, Bl)\n    annF = TH.build_anneal_full(geo, BY=int(by), num_warps=int(warps))\n    tot = int(n_temps) * (int(n_therm) + int(n_meas))\n    lad = {}\n    for nm, fn, a0 in (\n            ("produccion", lambda s: eng["anneal"](\n                s, jax.random.PRNGKey(42), btl, Pl, int(n_therm),\n                int(n_meas)), spl),\n            ("kernel", lambda s: annF(s, 42, btl, parl, k1l, int(n_therm),\n                                      int(n_meas)), TH.planar(spl))):\n        f = jax.jit(fn).lower(a0).compile()\n        t0 = time.perf_counter()\n        o = jax.block_until_ready(f(a0))\n        dt = time.perf_counter() - t0\n        lad[nm] = dict(wall_s=dt, us_per_replica_sweep=1e6 * dt / (tot * Bl))\n        print(f"  {nm:>11}: {dt:7.2f} s   "\n              f"{lad[nm][\'us_per_replica_sweep\']:7.3f} us/replica-barrido",\n              flush=True)\n        del o\n    lad["change_pct"] = 100 * (lad["kernel"]["wall_s"]\n                               / lad["produccion"]["wall_s"] - 1)\n    res["ladder"] = dict(n_temps=int(n_temps), n_therm=int(n_therm),\n                         n_meas=int(n_meas), replicas=Bl, total_sweeps=tot,\n                         arms=lad)\n    print(f"\\n  la escalera completa: {lad[\'change_pct\']:+.1f}%", flush=True)\n    res["ok"] = bool(res["exact_ok"] and res["physics_ok"])\n    if out:\n        json.dump(res, open(out, "w"), indent=1)\n        print(f"\\n  guardado {out}", flush=True)\n    return res\n\n\ndef run_states(reps=(50, 500), devices=(1, 2), n_temps=10, sweeps=20000,\n               Rd=18.30, L=5, by=4, warps=4, So=1.0, out=None,\n               label="kaggle-states"):\n    """One state and ten states, on one card and on two.\n\n    Kaggle hands out two T4s, which is exactly the pair this needs. The\n    one-device arm uses `jax.jit`, which places the whole batch on device 0;\n    the two-device arm uses `pmap`. Both replica counts divide by both device\n    counts (500 -> 500/250, 50 -> 50/25), so nothing has to be approximated\n    the way 50 across four cards did.\n\n    The ladder is SHORTENED on purpose. What this measures is a RATE --\n    microseconds per replica-sweep -- and the full 200-temperature clock is\n    reported as that rate times the real sweep count. A T4 running ten states\n    at full length would take hours, which is why the long cells are\n    extrapolated rather than waited for.\n    """\n    import jax\n    import jax.numpy as jnp\n    import sim_core as S\n\n    import time\n    kB = 0.086173404\n    n_therm = int(sweeps) * 2 // 3\n    n_meas = int(sweeps) - n_therm\n    geo = S.build_geometry(Rd=float(Rd), L=int(L))\n    eng = S.build_engine(geo)\n    surf = jnp.asarray(geo["surface"][None, ...])\n    T = np.linspace(20.0, 0.1, int(n_temps))\n    bt = jnp.asarray(1.0 / (kB * T), jnp.float32)\n    measured = int(n_temps) * (n_therm + n_meas)\n    full = 200 * (n_therm + n_meas)          # the production ladder\n    P0 = dict(Jex=1.0, Kan1=0.177, Kan2=0.0, KanS=0.139, Hex=0.483,\n              KDM=0.880, g=float(np.deg2rad(90.0)))\n\n    import triton_halo as TH\n\n    # Nothing is timed before the physics is proven in THIS run. A clock\n    # nobody checked does not belong on a chart, even when the same file\n    # verified elsewhere and its checksum matches.\n    chk = verify(geo, by=by, warps=warps, So=So)   # So now a real parameter\n    if not chk["ok"]:\n        print("\\n  la fisica no coincide: no se cronometra nada", flush=True)\n        if out:\n            json.dump(dict(label=label, env=environment(), verify=chk,\n                           ok=False, rows=[]), open(out, "w"), indent=1)\n        return dict(label=label, verify=chk, ok=False, rows=[])\n    print("", flush=True)\n\n    annF = TH.build_anneal_full(geo, BY=int(by), num_warps=int(warps), So=So)\n    avail = jax.local_device_count()\n    res = dict(label=label, env=environment(), n_temps=int(n_temps),\n               sweeps=int(sweeps), n_therm=n_therm, n_meas=n_meas,\n               measured_sweeps=measured, full_sweeps=full,\n               devices_available=avail, verify=chk, rows=[])\n    print(f"{avail} T4 visibles  |  escalera medida {n_temps}T x "\n          f"{n_therm}+{n_meas} = {measured:,} barridos", flush=True)\n    print(f"  el reloj a 200 T se reporta como tasa x {full:,} barridos",\n          flush=True)\n    print(f"\\n{\'disp\':>5}{\'repl\':>6}{\'/disp\':>7}{\'estandar\':>13}"\n          f"{\'optimizado\':>13}{\'cambio\':>9}", flush=True)\n\n    for nd in devices:\n        if nd > avail:\n            print(f"  {nd} dispositivos: solo hay {avail}, se omite", flush=True)\n            continue\n        for R in reps:\n            if R % nd:\n                continue\n            per = R // nd\n            c = lambda v: jnp.full((per, 1, 1, 1), float(v), jnp.float32)\n            P = {k: c(v) for k, v in P0.items()}\n            k1 = jnp.where(surf, P["KanS"], P["Kan1"])\n            par = TH.pack_params(P, per)\n            sp = S.init_spins(jax.random.PRNGKey(7), geo, R)\n            spP = TH.planar(sp)\n            if nd > 1:\n                sp = sp.reshape(nd, per, *sp.shape[1:])\n                spP = spP.reshape(nd, per, *spP.shape[1:])\n                keys = jax.random.split(jax.random.PRNGKey(42), nd)\n\n            row = dict(devices=nd, replicas=int(R), per_device=per)\n            for nm in ("estandar", "optimizado"):\n                if nm == "estandar":\n                    if nd == 1:\n                        f, a = jax.jit(lambda s: eng["anneal"](\n                            s, jax.random.PRNGKey(42), bt, P, n_therm,\n                            n_meas)), (sp,)\n                    else:\n                        f, a = jax.pmap(lambda s, k: eng["anneal"](\n                            s, k, bt, P, n_therm, n_meas)), (sp, keys)\n                else:\n                    g = lambda s: annF(s, 42, bt, par, k1, n_therm, n_meas)\n                    f, a = (jax.jit(g) if nd == 1 else jax.pmap(g)), (spP,)\n                try:                       # compile without executing\n                    f = f.lower(*a).compile()\n                except Exception:\n                    jax.block_until_ready(f(*a))\n                t0 = time.perf_counter()\n                o = jax.block_until_ready(f(*a))\n                dt = time.perf_counter() - t0\n                del o\n                us = 1e6 * dt / (measured * R)\n                row[nm] = dict(wall_s=dt, us_per_replica_sweep=us,\n                               full_ladder_s=us * full * R / 1e6)\n            row["change_pct"] = 100 * (row["optimizado"]["wall_s"]\n                                       / row["estandar"]["wall_s"] - 1)\n            res["rows"].append(row)\n            print(f"{nd:>5}{R:>6}{per:>7}"\n                  f"{row[\'estandar\'][\'full_ladder_s\']/60:>10.1f} min"\n                  f"{row[\'optimizado\'][\'full_ladder_s\']/60:>10.1f} min"\n                  f"{row[\'change_pct\']:>+8.1f}%", flush=True)\n\n    if out:\n        json.dump(res, open(out, "w"), indent=1)\n        print(f"\\n  guardado {out}", flush=True)\n    return res\n\n\ndef verify(geo, by=4, warps=4, So=1.0, batch=8, n_eq=400, temps=(20.0, 5.0, 1.0)):\n    """Prove the physics before any clock is believed, in seconds.\n\n    This exists because `run_states` used to time four cells and report them\n    with no physics evidence of its own: the verification lived in a different\n    run, and the argument reduced to "the same file ran there, and its sha256\n    matches". That is a real chain of evidence but it is not a measurement,\n    and a number nobody checked in the run that produced it should not be on\n    a chart. These checks cost seconds against tens of minutes of stopwatch.\n\n    Two tests, in the order that matters:\n\n      1. the stencil against `sim_core.substep` on IDENTICAL randomness, which\n         must give ZERO differing spins -- an exactness test, no statistics;\n      2. both engines to equilibrium with their OWN randomness, whose energies\n         must agree within the spread the run itself reports, which is what\n         catches a degenerate proposal that accepts everything or nothing.\n    """\n    import jax\n    import jax.numpy as jnp\n    import sim_core as S\n    import triton_halo as TH\n\n    kB = 0.086173404\n    beta = 1.0 / (kB * 5.0)\n    eng = S.build_engine(geo)\n    surf = jnp.asarray(geo["surface"][None, ...])\n    ev = jnp.asarray(geo["even"][None, ..., None])\n    od = jnp.asarray(geo["odd"][None, ..., None])\n    n_act = float(geo["n_active"])\n    P0 = dict(Jex=1.0, Kan1=0.177, Kan2=0.0, KanS=0.139, Hex=0.483,\n              KDM=0.880, g=float(np.deg2rad(90.0)))\n    mk = lambda b: {k: jnp.full((b, 1, 1, 1), float(v), jnp.float32)\n                    for k, v in P0.items()}\n\n    out = dict(exact=[], physics=[])\n    print("=== exactitud: el stencil contra sim_core.substep, "\n          "misma aleatoriedad ===", flush=True)\n    B = int(batch)\n    P = mk(B)\n    k1 = jnp.where(surf, P["KanS"], P["Kan1"])\n    sp = S.init_spins(jax.random.PRNGKey(3), geo, B)\n    kt, ka = jax.random.split(jax.random.PRNGKey(101))\n    tr = jax.random.normal(kt, sp.shape)\n    tr = tr / jnp.linalg.norm(tr, axis=-1, keepdims=True) * So\n    ru = jax.random.uniform(ka, sp.shape[:-1])\n    sub = TH.build(geo, BY=int(by), num_warps=int(warps), So=So)\n    par = TH.pack_params(P, B)\n    spP, trP = TH.planar(sp), TH.planar(tr)\n\n    an = lambda s, kk: (kk * ((s[..., 0] * s[..., 1]) ** 2\n                              + (s[..., 0] * s[..., 2]) ** 2\n                              + (s[..., 1] * s[..., 2]) ** 2)\n                        + P["Kan2"] * (s[..., 0] * s[..., 1] * s[..., 2]) ** 2)\n    for parity, mask, nm in ((0, ev, "par"), (1, od, "impar")):\n        trm = jnp.where(mask, tr, sp)\n        dS = trm - sp\n        xp = jnp.roll(sp, -1, 3); xm = jnp.roll(sp, 1, 3)\n        yp = jnp.roll(sp, -1, 2); ym = jnp.roll(sp, 1, 2)\n        zp = jnp.roll(sp, -1, 1).at[:, -1].set(0.0)\n        zm = jnp.roll(sp, 1, 1).at[:, 0].set(0.0)\n        dE = -P["Jex"] * jnp.sum(dS * (xp + xm + yp + ym + zp + zm), -1)\n        dE = dE + an(trm, k1) - an(sp, k1)\n        dE = dE - P["KDM"] * (dS[..., 1] * (xp[..., 2] - xm[..., 2])\n                              - dS[..., 2] * (xp[..., 1] - xm[..., 1])\n                              + dS[..., 2] * (yp[..., 0] - ym[..., 0])\n                              - dS[..., 0] * (yp[..., 2] - ym[..., 2]))\n        dE = dE - P["Hex"] * (dS[..., 0] * jnp.cos(P["g"])\n                              + dS[..., 2] * jnp.sin(P["g"]))\n        acc = ((dE < 0) | (ru < jnp.exp(-dE * beta)))[..., None] & mask\n        ref = jnp.where(acc, trm, sp)\n        got, dE_t = sub(spP, k1, par, beta, 0, parity, trP, ru, want_de=True)\n        sel = np.broadcast_to(np.asarray(mask[..., 0]), dE.shape)\n        mad = float(np.max(np.abs(np.asarray(dE)[sel]\n                                  - np.asarray(dE_t)[sel])))\n        bad = int(np.sum(np.asarray(ref) != np.asarray(TH.unplanar(got))))\n        tot = int(np.asarray(ref).size)\n        out["exact"].append(dict(sublattice=nm, dE_max_abs=mad,\n                                 spin_mismatch=bad, spin_total=tot))\n        print(f"  {nm:>5}: dE max |dif| {mad:.3e}   espines distintos "\n              f"{bad}/{tot}", flush=True)\n\n    print("\\n=== fisica: las dos cadenas a equilibrio, cada una con su RNG ===",\n          flush=True)\n    Bp = 16\n    Pp = mk(Bp)\n    k1p = jnp.where(surf, Pp["KanS"], Pp["Kan1"])\n    parp = TH.pack_params(Pp, Bp)\n    spE = S.init_spins(jax.random.PRNGKey(11), geo, Bp)\n    subp = TH.build(geo, BY=int(by), num_warps=int(warps), So=So)\n    for T in temps:\n        bT = 1.0 / (kB * T)\n\n        def jx(s):\n            def body(i, v):\n                st, k = v\n                k, a, b = jax.random.split(k, 3)\n                st = eng["substep"](st, a, bT, ev, Pp, k1p)\n                st = eng["substep"](st, b, bT, od, Pp, k1p)\n                return st, k\n            return jax.lax.fori_loop(0, int(n_eq), body,\n                                     (s, jax.random.PRNGKey(77)))[0]\n\n        sj = jax.block_until_ready(jax.jit(jx)(spE))\n        st = TH.unplanar(jax.block_until_ready(jax.jit(\n            TH.sweeps(subp, k1p, parp, bT, int(n_eq)))(TH.planar(spE))))\n        ej = float(jnp.mean(eng["energy"](sj, Pp, k1p)) / n_act)\n        et = float(jnp.mean(eng["energy"](st, Pp, k1p)) / n_act)\n        sd = float(jnp.std(eng["energy"](sj, Pp, k1p)) / n_act)\n        nrm = float(jnp.max(jnp.abs(\n            jnp.linalg.norm(st, axis=-1)\n            - jnp.asarray(geo["disco"][None]).astype(jnp.float32))))\n        row = dict(T=T, E_prod=ej, E_halo=et, E_sd=sd,\n                   sigma=abs(et - ej) / (sd + 1e-30), norm_err=nrm,\n                   M_prod=float(jnp.mean(eng["magnetization"](sj)[0])),\n                   M_halo=float(jnp.mean(eng["magnetization"](st)[0])))\n        out["physics"].append(row)\n        print(f"  T={T:>5.1f}  E/sitio {ej:+9.5f} / {et:+9.5f}  "\n              f"({row[\'sigma\']:.2f} sigma)   |M| {row[\'M_prod\']:.4f} / "\n              f"{row[\'M_halo\']:.4f}   |norma-1| {nrm:.1e}", flush=True)\n\n    out["exact_ok"] = all(r["spin_mismatch"] == 0 for r in out["exact"])\n    out["physics_ok"] = all(r["sigma"] < 3.0 and r["norm_err"] < 1e-5\n                            for r in out["physics"])\n    out["ok"] = bool(out["exact_ok"] and out["physics_ok"])\n    print(f"\\n  -> {\'la fisica coincide\' if out[\'ok\'] else \'AVISO: la fisica NO coincide\'}",\n          flush=True)\n    return out\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "0c115e3ce8af2a74ac7a1da682be3d02bae4e44c4d4374238e8b8b66e764d003", "triton_bench.py alterado"
open("triton_bench.py", "w").write(_src)
print("triton_bench.py  {} bytes  sha256 ok".format(len(_src)))


## 4 · Lectura y verificación de los parámetros

In [ ]:
# Lee los parámetros, cuenta los estados y verifica lo que no es negociable.
import numpy as np

v2 = lambda n: (n & -n).bit_length() - 1 if n else 0
P = np.asarray(PARAMETROS, dtype=float)
if P.ndim != 2 or P.shape[1] != 7:
    extra = ("  (¿sobra la temperatura? ya no es un parámetro: el "
             "enfriamiento es siempre 20 K -> 0.1 K)" if P.ndim == 2
             and P.shape[1] == 8 else "")
    raise SystemExit(f"PARAMETROS debe ser un arreglo de arreglos de 7 "
                     f"elementos; llegó con forma {P.shape}{extra}")

N_ESTADOS = P.shape[0]
nombres = list(NOMBRES) if NOMBRES else []
nombres += [f"estado-{i+1}" for i in range(len(nombres), N_ESTADOS)]
nombres = nombres[:N_ESTADOS]

print(f"{N_ESTADOS} estado(s) leídos de PARAMETROS")
print(f"{'':>3} {'estado':<12}{'Jex2':>7}{'Jex3':>7}{'Jex4':>7}"
      f"{'Kan1':>9}{'KanS':>8}{'Hex':>9}{'KDM':>9}")
for i, nm in enumerate(nombres):
    J2, J3, J4, K1, KS, H, D = P[i]
    print(f"{i+1:>3} {nm:<12}{J2:>7.3f}{J3:>7.3f}{J4:>7.3f}"
          f"{K1:>9.4f}{KS:>8.3f}{H:>9.4f}{D:>9.4f}")

# ---- las capas de vecinos activas, y el coloreo que piden
# Todos los estados de una corrida comparten el coloreo, así que las capas se
# toman como la unión sobre los estados: un Jex2 distinto de cero en un solo
# estado ya obliga a las ocho subredes para todos.
import sim_multishell as MS

activas = sorted({k for i in range(N_ESTADOS)
                  for k in MS.shells_of(*P[i, 0:3])})
COLOREO, N_SUBREDES, _f = MS.pick_colouring(activas)
print(f"\ncapas de vecinos activas: {activas}")
print(f"  coloreo: {COLOREO}  ->  {N_SUBREDES} subredes")
print(f"  verificado sobre los desplazamientos de cada capa: "
      f"{'si' if MS.check_colouring(_f, activas) else 'NO'}")
if N_SUBREDES > 2:
    print(f"  un barrido son {N_SUBREDES} pases y cada uno toca 1/{N_SUBREDES}"
          f" de los sitios:")
    print(f"  el trabajo por barrido es el mismo, el paralelismo por pase se"
          f" divide por {N_SUBREDES}")
    print(f"  y el kernel de Triton es de primera capa, así que se usará el"
          f" motor de JAX")

s = v2(REPLICAS) + v2(CAPAS)
print(f"\nv₂({REPLICAS}) + v₂({CAPAS}) = {v2(REPLICAS)} + {v2(CAPAS)} = {s}")
if s >= 2:
    print("  ⚠ con el motor 'estandar' este lote paga el castigo del ancho de")
    print(f"    vector. Con CAPAS={CAPAS}, una REPLICAS que no sea múltiplo de")
    print("    cuatro lo evita. El motor 'optimizado' no lo paga.")
else:
    print("  ✓ libre del castigo del ancho de vector en los dos motores.")

total = N_ESTADOS * REPLICAS
print(f"\ntrabajo: {N_ESTADOS} estados × {REPLICAS} réplicas = {total} cadenas")
print(f"         {N_TEMPERATURAS} temperaturas × {BARRIDOS} barridos "
      f"= {N_TEMPERATURAS*BARRIDOS:,} barridos por cadena")


## 5 · La corrida

In [ ]:
# Construye el motor y corre el enfriamiento de cada estado.
import time, json
import jax, jax.numpy as jnp
import sim_core as S

N_TERM = BARRIDOS * 2 // 3
N_MED  = BARRIDOS - N_TERM
# Con sólo la primera capa se usa `sim_core`, que es el motor verificado de
# referencia. Con capas superiores se usa `sim_multishell`, que lo reproduce
# EXACTAMENTE en ese caso (0 de 13500 espines distintos con la misma semilla)
# y además generaliza el coloreo.
if activas == [1]:
    geo = S.build_geometry(Rd=float(RD), L=int(CAPAS))
    eng = S.build_engine(geo, with_mvec=True)
    eng_mvec = True
else:
    geo = MS.build_geometry(Rd=float(RD), L=int(CAPAS), shells=activas)
    eng = MS.build_engine(geo)
    # el motor multicapa aun no emite el vector por replica
    eng_mvec = False
n_act = float(geo["n_active"])
print(f"red: N={geo['N']}  capas={geo['L']}  sitios activos={geo['n_active']}"
      f"  de {geo['L']*geo['N']**2} de la caja")

# ---- elegir el motor
usar_triton = False
if activas != [1] and MOTOR == "optimizado":
    raise SystemExit(f"el kernel optimizado cubre la primera capa de vecinos; "
                     f"con las capas {activas} usa MOTOR='auto' o 'estandar'")
if MOTOR in ("auto", "optimizado") and activas == [1]:
    try:
        import triton_halo as TH
        _ = TH.build_anneal_full(geo, BY=4, num_warps=4)
        usar_triton = True
    except Exception as e:
        if MOTOR == "optimizado":
            raise
        print(f"motor optimizado no disponible ({type(e).__name__}); se usa el estándar")
print("motor:", "optimizado (Triton)" if usar_triton else "estándar (JAX)")

# ---- verificación de física: el optimizado debe reproducir al estándar
if usar_triton:
    import triton_bench as TB
    chk = TB.verify(geo, by=4, warps=4)
    if not chk["ok"]:
        raise SystemExit("el motor optimizado no reproduce la física del "
                         "estándar: no se corre nada")

# ---- parámetros de un estado, difundidos a las réplicas
# El enfriamiento es el mismo para todos los estados, asi que se construye
# UNA vez y no uno por estado.
TEMPS = np.linspace(T_INICIAL, T_FINAL, int(N_TEMPERATURAS))
print(f"enfriamiento: {TEMPS[0]:.1f} K -> {TEMPS[-1]:.1f} K en "
      f"{len(TEMPS)} escalones de {TEMPS[0]-TEMPS[1]:.3f} K")


def columnas(fila, b):
    J2, J3, J4, K1, KS, H, D = fila
    c = lambda v: jnp.full((b, 1, 1, 1), float(v), jnp.float32)
    return dict(Jex=c(1.0), Jex2=c(J2), Jex3=c(J3), Jex4=c(J4),
                Kan1=c(K1), Kan2=c(0.0), KanS=c(KS),
                Hex=c(H), KDM=c(D), g=c(float(np.deg2rad(ANGULO))))

kB = 0.086173404
surf = jnp.asarray(geo["surface"][None, ...])
salida = {}
t_total = time.perf_counter()

# ---- cuantas GPU hay, y sobre que eje se reparten
#
# Esto existe porque la version anterior usaba `jax.jit` a secas, que coloca
# TODO en el dispositivo 0: en una maquina con dos T4 usaba una sola y la otra
# quedaba ociosa. Hay dos ejes sobre los que repartir y no son equivalentes.
#
#   estados   un estado entero por GPU. Es el eje que escala -- medido, 3.29x
#             sobre 4 GPU con 10 estados contra 1.10x con uno solo -- porque
#             cada estado lleva su propia escalera de temperaturas y sus
#             propios parametros, asi que no hay nada que comunicar.
#   replicas  parte las replicas de UN estado entre las GPU. Es lo que sirve
#             cuando hay menos estados que GPU. Medido en T4: 50 replicas en
#             2 T4 (25 por dispositivo) dan 3.413 us/replica-barrido contra
#             7.282 en una sola, es decir 2.13x.
#
# Los repartos DESIGUALES se serializan (+83.8% medido), asi que se exige
# divisibilidad exacta y, si no la hay, se usa una sola GPU y se dice por que.
n_dev = jax.local_device_count()
if n_dev > 1 and N_ESTADOS >= n_dev and N_ESTADOS % n_dev == 0:
    MODO, por_dev = "estados", REPLICAS
elif n_dev > 1 and REPLICAS % n_dev == 0:
    MODO, por_dev = "replicas", REPLICAS // n_dev
else:
    MODO, por_dev = "uno", REPLICAS
    if n_dev > 1:
        print(f"AVISO: {n_dev} GPU visibles, pero ni {N_ESTADOS} estados ni "
              f"{REPLICAS} replicas se dividen exacto entre {n_dev}; un "
              f"reparto desigual se serializa, asi que se usa 1 GPU")

print(f"GPU: {n_dev} visible(s) -> "
      f"{sorted({d.device_kind for d in jax.devices()})}")
if MODO == "estados":
    print(f"reparto: un estado por GPU, {N_ESTADOS // n_dev} tanda(s) "
          f"de {n_dev}, {REPLICAS} replicas cada uno")
elif MODO == "replicas":
    print(f"reparto: las {REPLICAS} replicas de cada estado entre {n_dev} "
          f"GPU, {por_dev} por dispositivo")
else:
    print(f"reparto: 1 GPU, {REPLICAS} replicas")


def _arma(i, b):
    """Lo que un dispositivo necesita para el estado `i` con `b` replicas."""
    Pc = columnas(P[i], b)
    temps = TEMPS
    return dict(Pcols=Pc, temps=temps,
                betas=jnp.asarray(1.0 / (kB * temps), jnp.float32),
                k1=jnp.where(surf, Pc["KanS"], Pc["Kan1"]),
                par=(TH.pack_params(Pc, b) if usar_triton else None))


mvectores = {}          # nombre -> (n_T, replicas, 3), o None


def _guarda(nm, obs, snap, temps, dt, repl, mvec=None):
    mvectores[nm] = None if mvec is None else np.asarray(mvec, np.float32)
    red = S.reduce_obs(np.asarray(obs), temps, n_act)
    salida[nm] = dict(T=temps.tolist(),
                      **{k: np.asarray(v).tolist() for k, v in red.items()
                         if k != "T"},
                      reloj_s=dt,
                      us_por_replica_barrido=1e6 * dt
                        / (int(N_TEMPERATURAS) * BARRIDOS * repl))
    salida[nm]["_snap"] = np.asarray(snap).tolist()
    print(f"  {nm:<12}{dt:7.1f} s   "
          f"{salida[nm]['us_por_replica_barrido']:6.3f} us/replica-barrido")


if MODO == "estados":
    # Un estado entero por dispositivo. La escalera, los parametros y la
    # semilla cambian por estado, asi que viajan como ejes del pmap.
    for base in range(0, N_ESTADOS, n_dev):
        idx = list(range(base, base + n_dev))
        pz = [_arma(i, REPLICAS) for i in idx]
        betas = jnp.stack([q["betas"] for q in pz])
        k1 = jnp.stack([q["k1"] for q in pz])
        sem = jnp.asarray([SEMILLA + i for i in idx], jnp.int32)
        sp = [S.init_spins(jax.random.PRNGKey(SEMILLA + i), geo, REPLICAS)
              for i in idx]
        if usar_triton:
            ann = TH.build_anneal_full(geo, BY=4, num_warps=4,
                                       with_mvec=True)
            par = jnp.stack([q["par"] for q in pz])
            g = lambda s, sd, bt, pr, kk: ann(s, sd, bt, pr, kk,
                                              N_TERM, N_MED)
            arg = (jnp.stack([TH.planar(x) for x in sp]), sem, betas, par, k1)
        else:
            Pst = {k: jnp.stack([q["Pcols"][k] for q in pz])
                   for k in pz[0]["Pcols"]}
            g = lambda s, sd, bt, pc: eng["anneal"](
                s, jax.random.PRNGKey(sd), bt, pc, N_TERM, N_MED)
            arg = (jnp.stack(sp), sem, betas, Pst)
        t0 = time.perf_counter()
        _o = jax.block_until_ready(jax.pmap(g)(*arg))
        # only the Triton engine returns the magnetisation vector
        obs, final, snaps, mvec = _o if (usar_triton or eng_mvec) \
            else (*_o, None)
        dt = time.perf_counter() - t0
        for j, i in enumerate(idx):
            _guarda(nombres[i], np.asarray(obs)[j],
                    np.asarray(snaps)[j][-1], pz[j]["temps"], dt, REPLICAS,
                    None if mvec is None else np.asarray(mvec)[j])

elif MODO == "replicas":
    # Las replicas de un estado repartidas. Todos los dispositivos comparten
    # la escalera y los parametros; solo cambian los espines y la semilla.
    for i, nm in enumerate(nombres):
        q = _arma(i, por_dev)
        sem = jnp.asarray([SEMILLA + i * 1000 + d for d in range(n_dev)],
                          jnp.int32)
        sp = S.init_spins(jax.random.PRNGKey(SEMILLA + i), geo, REPLICAS)
        sp = sp.reshape(n_dev, por_dev, *sp.shape[1:])
        if usar_triton:
            ann = TH.build_anneal_full(geo, BY=4, num_warps=4,
                                       with_mvec=True)
            g = lambda s, sd: ann(s, sd, q["betas"], q["par"], q["k1"],
                                  N_TERM, N_MED)
            arg = (jnp.stack([TH.planar(x) for x in sp]), sem)
        else:
            g = lambda s, sd: eng["anneal"](s, jax.random.PRNGKey(sd),
                                            q["betas"], q["Pcols"],
                                            N_TERM, N_MED)
            arg = (sp, sem)
        t0 = time.perf_counter()
        _o = jax.block_until_ready(jax.pmap(g)(*arg))
        # only the Triton engine returns the magnetisation vector
        obs, final, snaps, mvec = _o if (usar_triton or eng_mvec) \
            else (*_o, None)
        dt = time.perf_counter() - t0
        # `anneal` devuelve (n_T, 6, replicas) POR dispositivo, asi que las
        # replicas se reunen concatenando por el ULTIMO eje. Aplanar el eje
        # principal mezclaria temperaturas con observables, que es justo el
        # error que `reduce_obs` advierte en su docstring.
        obs = np.concatenate(list(np.asarray(obs)), axis=-1)
        # the vector is (n_dev, n_T, por_dev, 3); the replicas live on axis 1
        # of each device's slice, so they rejoin there and nowhere else
        mv = (None if mvec is None
              else np.concatenate(list(np.asarray(mvec)), axis=1))
        _guarda(nm, obs, np.asarray(snaps)[0][-1], q["temps"], dt, REPLICAS,
                mv)

else:
    for i, nm in enumerate(nombres):
        q = _arma(i, REPLICAS)
        sp = S.init_spins(jax.random.PRNGKey(SEMILLA + i), geo, REPLICAS)
        if usar_triton:
            ann = TH.build_anneal_full(geo, BY=4, num_warps=4,
                                       with_mvec=True)
            f = jax.jit(lambda s: ann(s, SEMILLA + i, q["betas"], q["par"],
                                      q["k1"], N_TERM, N_MED))
            arg = TH.planar(sp)
        else:
            f = jax.jit(lambda s: eng["anneal"](
                s, jax.random.PRNGKey(SEMILLA + i), q["betas"], q["Pcols"],
                N_TERM, N_MED))
            arg = sp
        f = f.lower(arg).compile()          # compila sin ejecutar
        t0 = time.perf_counter()
        _o = jax.block_until_ready(f(arg))
        obs, final, snaps, mvec = _o if (usar_triton or eng_mvec) \
            else (*_o, None)
        dt = time.perf_counter() - t0
        _guarda(nm, obs, snaps[-1], q["temps"], dt, REPLICAS, mvec)

print(f"\ntotal: {(time.perf_counter()-t_total)/60:.2f} min para "
      f"{N_ESTADOS} estado(s)")


## 6 · Las métricas físicas

In [ ]:
# Las métricas físicas, estado por estado, en su temperatura objetivo.
#
# Cómo se calcula cada una:
#
#   E/sitio   energía total del Hamiltoniano dividida por los sitios activos.
#             Cada enlace se cuenta UNA vez, usando sólo los desplazamientos
#             +x, +y y +z, nunca los seis.
#   |M|       norma del vector de magnetización promedio sobre los sitios
#             activos. Mz es su componente z.
#   Cv        capacidad calorífica, de la varianza de la energía:
#                 Cv = ⟨(ΔE)²⟩ / (k_B T² N)
#   χ         susceptibilidad, de la varianza de |M|:  χ = ⟨(Δ|M|)²⟩ / (k_B T)
#   Q         carga topológica de la capa media por la fórmula de
#             Berg–Lüscher, que es cuantizada — a diferencia de una diferencia
#             finita, que no lo es.
#
# Cv y χ son varianzas, y acumular E y E² en bruto las destruye en float32:
# a T = 0.2 una ulp de ⟨E²⟩ es mayor que la varianza misma y Cv sale negativa.
# Por eso se acumula la fluctuación respecto a una referencia tomada una vez
# por temperatura; un desplazamiento constante deja la varianza intacta.

print(f"{'estado':<12}{'T':>7}{'E/sitio':>11}{'|M|':>9}{'Mz':>9}"
      f"{'Cv':>10}{'χ':>10}{'Q':>9}")
for nm in nombres:
    r = salida[nm]
    j = -1                                   # el fondo de la escalera, 0.1 K
    print(f"{nm:<12}{r['T'][j]:>7.2f}{r['E'][j]:>11.5f}{r['M'][j]:>9.4f}"
          f"{r['Mz'][j]:>9.4f}{r['Cv'][j]:>10.4f}{r['chi'][j]:>10.4f}"
          f"{r['Q'][j]:>9.4f}")

# ══════════════════════════════════════════════════════════════════════════
#  LA TRANSICIÓN
# ══════════════════════════════════════════════════════════════════════════
#
# Con la escalera fija, la temperatura de transición deja de ser un dato de
# entrada y pasa a ser algo que se mide. Se localiza por el PICO de las dos
# funciones de respuesta, que es la definición operativa habitual en un
# sistema finito:
#
#   Cv = ⟨(ΔE)²⟩ / (k_B T² N)     pico -> transición de ORDENAMIENTO
#   χ  = ⟨(Δ|M|)²⟩ / (k_B T)      pico -> transición MAGNÉTICA
#
# Tres advertencias que hay que leer junto al número:
#
#   · La malla es de 0.1 K, así que el pico no se puede situar mejor que eso.
#     Aquí NO se interpola: un ajuste parabólico sobre tres puntos ruidosos
#     da una precisión aparente que el dato no tiene.
#   · Esto es un sistema FINITO (5245 sitios). Lo que se ve es un máximo
#     redondeado, no una divergencia, y su posición se desplaza con el tamaño:
#     no es la T_c del límite termodinámico.
#   · Si el pico cae en un extremo de la escalera, la transición está FUERA
#     del rango recorrido y el número no significa nada. Se avisa.

print()
print(f"{'estado':<12}{'T(pico Cv)':>12}{'Cv máx':>10}"
      f"{'T(pico χ)':>12}{'χ máx':>10}  aviso")
transiciones = {}
for nm in nombres:
    r = salida[nm]
    T = np.asarray(r["T"])
    fila = {}
    msg = []
    for k, etiq in (("Cv", "Cv"), ("chi", "chi")):
        v = np.asarray(r[k], dtype=float)
        v = np.where(np.isfinite(v), v, -np.inf)
        j = int(np.argmax(v))
        fila[f"T_{etiq}"] = float(T[j])
        fila[f"{etiq}_max"] = float(v[j])
        if j == 0 or j == len(T) - 1:
            msg.append(f"el pico de {etiq} cae en el borde")
    transiciones[nm] = fila
    print(f"{nm:<12}{fila['T_Cv']:>12.1f}{fila['Cv_max']:>10.4f}"
          f"{fila['T_chi']:>12.1f}{fila['chi_max']:>10.4f}"
          f"  {'; '.join(msg) if msg else 'ok'}")

print()
print(f"malla de {TEMPS[0]-TEMPS[1]:.1f} K, asi que cada pico vale "
      f"+-{(TEMPS[0]-TEMPS[1])/2:.2f} K; sistema finito de "
      f"{int(n_act)} sitios, el maximo es redondeado y no una divergencia")

with open("resultados.json", "w") as f:
    json.dump(dict(parametros=P.tolist(), nombres=nombres,
                   geometria=dict(Rd=RD, capas=CAPAS, N=int(geo["N"]),
                                  sitios=int(geo["n_active"])),
                   cronograma=dict(replicas=REPLICAS, n_temperaturas=N_TEMPERATURAS,
                                   barridos=BARRIDOS, n_term=N_TERM, n_med=N_MED),
                   motor="optimizado" if usar_triton else "estandar",
                   resultados=salida, transiciones=transiciones), f, indent=1)
print("\nguardado resultados.json")

# ══════════════════════════════════════════════════════════════════════════
#  LOS ESTADOS, UN .npz POR ESTADO
# ══════════════════════════════════════════════════════════════════════════
#
# Hasta aqui lo unico que sobrevivia a la corrida eran las curvas YA
# promediadas sobre replicas, mas una sola imagen (la capa media de la replica
# 0 en la ultima temperatura). El registro por replica se perdia.
#
# Cada archivo lleva:
#
#   parametros    (7,)           los siete del estado, en el orden de
#                                PARAMETROS: Jex2, Jex3, Jex4, Kan1, KanS,
#                                Hex, KDM
#   M             (50, 200, 3)   la magnetizacion media, indexada
#                                [replica, temperatura, componente]
#   temperaturas  (200,)         la escalera, que indexa el eje 1 de M
#
# La forma es (replicas, temperaturas, 3) y no una lista plana de 10000 filas
# a proposito: asi no hay ningun orden que documentar. M[r] es la curva de
# enfriamiento COMPLETA de la replica r, M[:, t] son las 50 replicas a la
# temperatura t, y temperaturas[t] dice cual es esa temperatura. Una lista
# plana obliga a recordar si el orden era por temperatura o por replica, y ese
# es justo el tipo de detalle que se pierde entre quien escribe el archivo y
# quien lo lee.
#
# Que es M. No es una configuracion instantanea: es el promedio sobre los
# N_MED barridos de medicion de esa temperatura, para esa replica. Es decir un
# promedio termico, uno por replica, no una muestra suelta.

import os

guardados = []
for nm in nombres:
    mv = mvectores.get(nm)
    if mv is None:
        print(f"  {nm}: sin vector de magnetizacion "
              f"(lo da el motor optimizado, y aqui corrio el estandar)")
        continue
    i = nombres.index(nm)
    nT, nR, _ = mv.shape
    # el motor lo entrega (temperatura, replica, 3); se transpone, no se
    # aplana, para que el archivo no tenga un orden que haya que explicar
    M = np.transpose(mv, (1, 0, 2)).astype(np.float32)
    fn = f"estado_{nm}.npz"
    np.savez_compressed(
        fn,
        parametros=np.asarray(P[i], np.float32),
        M=M,
        temperaturas=np.asarray(salida[nm]["T"], np.float32))
    guardados.append(fn)
    print(f"  {fn}  parametros (7,)  M {M.shape} "
          f"[replica, temperatura, componente]  "
          f"{os.path.getsize(fn)/1024:.0f} KB")

if guardados:
    print(f"\nguardados {len(guardados)} .npz")


## 7 · Las figuras

In [ ]:
# Las curvas termodinámicas de cada estado al enfriar.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 2, figsize=(11, 7))
for nm in nombres:
    r = salida[nm]
    for a, k, t in ((ax[0, 0], "E", "energía por sitio"),
                    (ax[0, 1], "M", "|M|"),
                    (ax[1, 0], "Cv", "Cv"),
                    (ax[1, 1], "chi", "χ")):
        a.plot(r["T"], r[k], lw=1.4, label=nm)
        a.set_xlabel("T (K)"); a.set_ylabel(t)
        a.grid(alpha=.25); a.invert_xaxis()
ax[0, 0].legend(fontsize=9)
fig.suptitle(f"{N_ESTADOS} estado(s) · {REPLICAS} réplicas · "
             f"{N_TEMPERATURAS}×{BARRIDOS} barridos", y=1.0)
plt.tight_layout(); plt.show()

# La configuración final de la capa media, una réplica por estado.
fig, axs = plt.subplots(1, N_ESTADOS, figsize=(3.1 * N_ESTADOS, 3.3))
axs = np.atleast_1d(axs)
for a, nm in zip(axs, nombres):
    im = a.imshow(np.asarray(salida[nm].get("_snap", np.zeros((geo["N"], geo["N"])))),
                  cmap="jet", vmin=-1, vmax=1)
    a.set_title(f"{nm}  ($s_z$)", fontsize=10); a.axis("off")
fig.colorbar(im, ax=axs, fraction=.025)
plt.show()
